# Qwen3.5 Financial Analyst with LoRA for project 24

[19. Forecasting and Kelly Allocation](https://ramtin-asadi.github.io/Quantitative-Finance-Lab/notebooks/19_forecasting_kelly_allocation.html) and [20. Reinforcement Learning Portfolio Allocation](https://ramtin-asadi.github.io/Quantitative-Finance-Lab/notebooks/20_rl_portfolio_allocation.html) already gave us the deep-learning prerequisites: tensors, embeddings, backpropagation, optimizers, recurrent and convolutional sequence models, regularization, and the basic idea that a neural network learns a parameterized function from data. We can use that knowledge rather than starting again from the chain rule.

Here we move into **large language models** and train the language component that the financial-analysis system will use later. The path is:

- build the probability model behind an autoregressive LLM and then open the transformer itself;
- understand the pinned Qwen3.5-2B architecture, including its mixture of full attention and Gated DeltaNet layers;
- turn reviewed financial examples into completion-only supervised language-model targets;
- fine-tune only small low-rank adapters while the pretrained base stays frozen;
- validate the adapter with financial evidence, citation, schema, numerical-traceability and human-review checks;
- merge the adapter back into the base model, convert the result to GGUF, quantize it to Q4_K_M, and validate the exact local artifact we will deploy.

The distinction between **learning** and **deployment** stays important throughout the training workflow. During training we use the original 16-bit base model and float32 LoRA parameters. The 4-bit artifact appears only after training, merging, conversion and quantization.

The official [Qwen3.5-2B model card](https://huggingface.co/Qwen/Qwen3.5-2B), the original [Transformer paper](https://arxiv.org/abs/1706.03762), the [LoRA paper](https://arxiv.org/abs/2106.09685), and the [Gated DeltaNet paper](https://arxiv.org/abs/2412.06464) are useful references for the architecture we unpack below.

> We are training a narrow financial-analysis behavior, not trying to recreate Qwen pretraining. Most of the model's language, reasoning and world representation already live in the pretrained weights. Our data teach the model how to use supplied financial evidence, cite it, keep numbers traceable and return a constrained analyst-style answer.

## 1. From Deep Learning to a Causal Language Model

A language model assigns probabilities to sequences of tokens. If the tokenized sequence is

$$
x_1,x_2,\ldots,x_T,
$$

the chain rule of probability lets us write the probability of the complete sequence as

$$
p(x_1,\ldots,x_T)=\prod_{t=1}^{T}p(x_t\mid x_1,\ldots,x_{t-1}).
$$

That factorization is the foundation of a causal LLM. At token position $t$, the model sees only the prefix and predicts a distribution for the next token. During generation, the predicted token becomes part of the next prefix and the process repeats.

The word *token* is important. The model doesn't read text as human words. A tokenizer maps text into integer IDs from a fixed vocabulary. Common words may be one token, rare words may split into pieces, punctuation has tokens, and numbers can split in ways that look unintuitive. The tokenizer used by the pinned Qwen model therefore belongs to the model itself. Changing the tokenizer while keeping the weights fixed would change the meaning of the embedding rows the network learned during pretraining.

For a vocabulary of size $V$, the last language-model layer produces a vector of **logits**

$$
z_t\in\mathbb{R}^{V}.
$$

A softmax turns those scores into probabilities:

$$
p(x_{t+1}=j\mid x_{\le t})=\frac{\exp(z_{t,j})}{\sum_{k=1}^{V}\exp(z_{t,k})}.
$$

If the correct next token is $y_t$, the one-token negative log-likelihood is

$$
\ell_t=-\log p(y_t\mid x_{<t}).
$$

Over supervised positions, training minimizes the mean or sum of these losses. A probability of $0.9$ for the right token gives a small penalty; a probability of $0.001$ gives a large one. Nothing in this objective explicitly says “understand credit,” “cite the source,” or “do financial analysis.” Those behaviors have to emerge from pretraining and from the supervised examples and constraints we impose later.

### Tokens become vectors

An integer token ID has no geometry by itself. An embedding matrix

$$
E\in\mathbb{R}^{V\times d}
$$

maps token $x_t$ to a $d$-dimensional vector

$$
h_t^{(0)}=E[x_t].
$$

Project 19 already used embeddings as learned lookup tables. In an LLM, the same idea scales to a very large vocabulary and a long sequence. The embedding is the initial point in the model's **residual stream**. Each transformer block reads that stream, computes a contextual update and adds information back into it.

A useful mental model is that the network keeps rewriting each token's representation. The embedding for “spread” initially reflects its broad learned language meaning. After enough layers, “spread” in “high-yield spread widened 40 bp” can carry information about the surrounding issuer, units, market, date and earlier evidence in the prompt.

### Prompting and generation use the same model

There is no separate “question module” and “answer module.” A chat prompt is serialized into one token sequence using Qwen's chat template. The model then keeps doing next-token prediction at the assistant boundary. Fine-tuning changes the conditional distribution so that, when the prefix looks like our analyst prompt, high probability moves toward the structured, evidence-grounded completion we want.

That gives us the first connection between training and the later Project 24 system:

$$
\text{prompt evidence} \longrightarrow p_\theta(\text{analyst completion}\mid\text{prompt evidence}).
$$

The later retrieval system decides **what evidence enters the prefix**. LoRA training changes how the model uses that prefix.

### 1.1 From logits back to financial language

It is easy to look at an LLM answer and imagine that the model first forms a complete paragraph internally and then prints it. Generation is more local than that.

Suppose the current prefix ends with:

> Operating cash flow fell 52% while net income ...

The model computes a distribution for one next token. Tokens completing “rose,” “increased,” a number, punctuation or another phrase compete in the vocabulary. After one token is selected, the complete transformer runs again for the next token, using cached sequence state where possible.

A multi-sentence financial argument is therefore built through a long sequence of locally conditioned decisions. High-level coherence emerges because hidden states encode information from the entire prefix and because pretraining exposed the model to enormous numbers of coherent sequences.

This has two practical consequences.

First, **early wording choices constrain later text**. If the model starts a claim with “The source reports,” it becomes more likely to continue in an extractive factual style. If it starts “This suggests,” it moves toward interpretation. Our schema's explicit `kind` field helps set that local trajectory.

Second, **structured outputs reduce the search space**. After generating `"materiality":`, legal enum values are few. After `"evidence_ids":`, constrained generation can permit only known IDs. A model with modest parameter count becomes more reliable when the application gives it strong local structure.

### Internal representation isn't a database row

The residual vector at one token is a distributed representation. There is no single “CPI neuron” we query like a SQL column. Features are spread across dimensions, heads and layers.

That is another reason we prefer structured context for exact numbers. The model can reason over a supplied value such as `2Y yield change = +7 bp` without needing that value to be encoded permanently in weights.

Weights are best used for patterns and transformations. External context is better for date-sensitive facts.

### 1.2 Attention is learned routing, not a human explanation

People often visualize attention weights and call them explanations. Mathematically, attention weights are coefficients in one internal aggregation:

$$
h_i=\sum_j A_{ij}v_j.
$$

A large $A_{ij}$ says head $h$ at position $i$ places more weight on value vector $v_j$ in that layer. It doesn't prove that token $j$ caused the final financial conclusion in a human-interpretable sense. Later MLPs, other heads, residual paths and subsequent layers can transform the information.

For our analyst, source citations are therefore much more useful than raw attention maps as an audit mechanism. The answer says which evidence IDs it relies on, and deterministic validators can inspect those references.

We use the transformer to compute; we use the evidence architecture to explain where factual support came from.

### Attention head specialization is emergent

No line of code declares that one head will track entity names and another will compare numbers. Heads learn whatever features reduce pretraining and fine-tuning loss.

LoRA changes the Q/K/V/O projections, so it can shift these learned routing patterns. But we don't assign semantic roles to individual heads in the training notebook because we haven't run an interpretability study that supports such labels.

That restraint is useful. A good educational notebook should explain the mechanism fully without inventing post-hoc stories about internal units we didn't measure.

### 1.3 Causal masking: why future answer tokens can't leak backward

A transformer processes many positions in parallel during training. Without a restriction, token $t$ could attend to token $t+1$, read the correct answer from the future and make the objective meaningless.

Causal attention applies a triangular mask. In the full-attention layers, the unnormalized attention score between query position $i$ and key position $j$ is

$$
s_{ij}=\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$

We add a causal mask $M$,

$$
M_{ij}=
\begin{cases}
0, & j\le i,\\
-\infty, & j>i,
\end{cases}
$$

and compute

$$
A_{ij}=\operatorname{softmax}_j(s_{ij}+M_{ij}).
$$

Future positions receive zero probability after softmax. Training can therefore evaluate all answer-token losses in one forward pass while respecting the same information restriction used in left-to-right generation.

The causal mask is different from the **loss mask** we will use later. Causal masking controls what each position can *see*. Completion-only masking controls which positions *contribute to the optimization objective*. In our dataset, prompt tokens are visible to the answer, but prompt positions are assigned label `-100`, so the trainer doesn't spend gradient on reproducing the user prompt.

### 1.4 The transformer block: attention, residual paths and the feed-forward network

A transformer block repeatedly performs two jobs:

1. exchange information across token positions;
2. transform the representation at each position.

In a simplified pre-normalization form,

$$
u = h + \operatorname{Attention}(\operatorname{Norm}(h)),
$$

$$
h' = u + \operatorname{MLP}(\operatorname{Norm}(u)).
$$

The residual additions are central. Instead of forcing every layer to replace the representation, the layer learns an update. A deep network can therefore carry useful information forward while adding increasingly contextual features.

Qwen uses RMS-style normalization. For a vector $x\in\mathbb{R}^d$,

$$
\operatorname{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon},
$$

and a learned scale vector $g$ gives

$$
\operatorname{RMSNorm}(x)=g\odot \frac{x}{\operatorname{RMS}(x)}.
$$

Unlike LayerNorm, RMSNorm doesn't subtract the feature mean. Its job is still familiar from earlier deep-learning projects: keep activation scale controlled enough that deep residual computation can train stably.

The feed-forward part expands the hidden representation, applies a gated nonlinearity and projects it back. A common gated form used by Qwen-family models can be written

$$
\operatorname{FFN}(x)=W_{\text{down}}\left[\operatorname{SiLU}(W_{\text{gate}}x)\odot (W_{\text{up}}x)\right].
$$

`gate_proj`, `up_proj` and `down_proj` later appear directly in our LoRA target list. Their names reflect these three matrices. The gate controls which transformed features pass; the up projection creates a larger intermediate feature space; the down projection returns to the residual-stream width.

When our financial LoRA targets these MLP projections, it can adjust more than word-to-word attention. It can also alter the nonlinear feature transformations that map retrieved evidence into analyst-style internal representations.

### 1.5 Full attention, heads, grouped-query attention and position

For one full-attention layer, hidden states $X$ are projected into queries, keys and values:

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
$$

The query asks what information a token needs. Keys determine how other positions can be matched. Values carry the information that gets aggregated. In one attention head,

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
$$

**Multi-head attention** repeats this operation in several learned subspaces. A head can specialize in a different relationship: entity continuity, quotation boundaries, numerical references, long-distance syntax, or any other pattern that training finds useful.

Qwen3.5-2B's published configuration uses grouped-query attention in its full-attention layers: more query heads than key/value heads. Queries can therefore stay diverse while groups share K/V projections. At inference this reduces the size of the K/V cache, which is important when a local model is asked to read thousands of financial-context tokens.

Position also has to enter the computation. Attention by itself is permutation-equivariant: if we shuffled tokens and shuffled their vectors consistently, it would have no intrinsic sense of order. Rotary position embeddings, or **RoPE**, rotate pairs of query/key coordinates by position-dependent angles. For one two-dimensional pair,

$$
R(\theta_t)
=
\begin{bmatrix}
\cos\theta_t & -\sin\theta_t\\
\sin\theta_t & \cos\theta_t
\end{bmatrix},
$$

and the position-aware query becomes

$$
\tilde q_t=R(\theta_t)q_t.
$$

Keys receive the same type of rotation. Dot products then depend naturally on relative position. We don't need to reconstruct every implementation detail to understand the consequence: the same word at token 50 and token 5,000 carries different positional phase, allowing attention to reason over order and distance.

For the analyst later, this helps a claim near the end of a prompt retrieve a numerical fact many tokens earlier. It doesn't guarantee perfect long-context recall. That is why Project 24 still uses retrieval, evidence packing and token budgets rather than throwing an unlimited document archive into the prompt.

### 1.6 Qwen3.5-2B is a hybrid sequence model

The pinned base is `Qwen/Qwen3.5-2B`. Its published configuration has a 2,048-dimensional text hidden state and 24 text layers. The layer pattern repeats **three linear-attention layers followed by one full-attention layer**. Six such groups give the 24-layer stack.

The linear layers are based on **Gated DeltaNet**, which belongs to a different family from ordinary softmax attention. Full attention explicitly compares a query against stored keys from many earlier positions, giving strong content-addressable retrieval but an attention computation and K/V cache that grow with context length. Delta-style linear attention instead maintains a recurrent memory state that can be updated as tokens arrive.

A schematic delta-rule memory update can be written as

$$
S_t = \alpha_t S_{t-1}
+
\beta_t\left(v_t-\alpha_t S_{t-1}k_t\right)k_t^\top,
$$

where:

- $S_t$ is the compressed memory state;
- $\alpha_t$ controls forgetting or retention of the previous state;
- $\beta_t$ controls the strength of the current write;
- $k_t$ determines where the write is addressed;
- $v_t$ is the value we want memory to recover.

The residual term

$$
v_t-\alpha_tS_{t-1}k_t
$$

asks how different the desired value is from what the current memory would already return at key $k_t$. The update therefore behaves like a learned corrective write rather than simply accumulating every past token.

Qwen mixes these linear-memory layers with periodic full-attention layers. We can view the design as a compromise: frequent recurrent-style sequence processing gives efficient long-context state updates, while regular full-attention blocks restore explicit token-to-token retrieval capacity.

The model also includes a vision encoder because the architecture is multimodal, which is why the training code uses Unsloth's `FastVisionModel`. We are building a text financial analyst. The training configuration explicitly freezes visual layers and visual projectors, and LoRA is enabled only for the language model. No screenshot, chart image or visual token is part of this fine-tuning dataset.

The base model's native architecture can handle contexts much longer than the 8,192-token training window chosen here. We deliberately train at 8,192 because the supervised examples have bounded evidence packets, GPU memory is finite, and changing the full adaptation process around very long contexts would increase cost. The later local application uses its own larger runtime context and validates how the exported model behaves there.

### 1.7 Gated DeltaNet in slower motion

The recurrent memory equation deserves a more intuitive derivation because it is one of the main differences between this Qwen generation and the vanilla transformer from *Attention Is All You Need*.

Suppose a memory matrix $S$ should map a key-like vector $k$ to a value-like vector $v$:

$$
Sk\approx v.
$$

If the current memory predicts

$$
\hat v=S_{t-1}k_t
$$

for the new key $k_t$, the prediction error is

$$
e_t=v_t-\hat v.
$$

A delta rule can write a correction proportional to the outer product

$$
\Delta S_t=\beta_t e_t k_t^\top.
$$

Then

$$
S_t=S_{t-1}+\Delta S_t.
$$

The outer product creates a matrix update that is strongest in the direction associated with $k_t$. If the old memory already returns the desired $v_t$, the residual is small and little needs to be written.

Gated DeltaNet adds learned controls over retention and writing. A retention/forget gate can shrink older state:

$$
\tilde S_{t-1}=\alpha_tS_{t-1},
$$

and the write becomes

$$
S_t=
\tilde S_{t-1}
+
\beta_t(v_t-\tilde S_{t-1}k_t)k_t^\top.
$$

Now the model can decide both how much old memory to carry forward and how strongly to correct it.

This is conceptually related to recurrent gating we saw around GRU/LSTM models in Project 19, but the state is designed to behave like an associative key-value memory rather than a single hidden vector. We shouldn't collapse the architectures into each other; the useful connection is that both learn **state updates** instead of explicitly retaining all previous token pairs.

### Why periodic softmax attention still helps

Compressed state creates interference. Two different historical patterns can write into overlapping memory directions. Full attention doesn't have to compress earlier positions into one fixed state: it can compare the current query with a sequence of stored keys.

Qwen therefore alternates the mechanisms. The model can repeatedly summarize context through efficient recurrent memory, then use a full-attention layer to perform explicit content-addressed retrieval.

For financial prompts, imagine dozens of evidence IDs and numbers. A recurrent layer can track broad state such as “cash conversion weakening,” while a full-attention layer can still look back to the exact evidence token containing `52.18 percent`.

That description is an intuition for the mechanisms, not a claim that we measured individual Qwen layers doing exactly those jobs.

### 1.8 Autoregressive inference reuses state instead of recomputing the whole prefix

If we naively generated token $T+1$, then reran all $T+1$ positions from scratch to generate token $T+2$, decoding would be extremely wasteful.

Full-attention layers cache key/value projections for earlier positions. At a new token, we compute only the new query/key/value and attend the query to the cached history.

For one full-attention layer, the cache contains something like

$$
K_{1:T},V_{1:T}.
$$

After producing token $T+1$, we append

$$
K_{T+1},V_{T+1}.
$$

Linear/recurrent layers similarly carry their recurrent state forward.

This is why prompt processing and generation have very different performance profiles. During prompt prefill, many tokens can be processed in parallel. During decoding, every next token depends on the previous generated token even though layer state is cached.

Project 24 later calibrates how many Qwen layers can be offloaded to a 2 GB MX450 and measures both prompt and generation throughput. That runtime experiment is the deployment consequence of the architecture we are studying here.

### Context length versus information quality

A model can technically accept a long window while still struggling to use every detail equally well. Retrieval systems exist partly because context is a scarce **attention and relevance budget**, not only a hard token limit.

If we give the model 20,000 tokens containing ten relevant lines and thousands of irrelevant lines, we increase compute and make evidence selection harder. If we retrieve 4,000 highly relevant tokens with dated source IDs, the conditional prediction problem becomes cleaner.

So the training notebook teaches the model how to use evidence. The application notebook teaches the system how to **choose evidence worth giving it**.

### 1.9 The exact Qwen3.5-2B language stack we are adapting

It helps to put concrete dimensions around the previous equations. The published Qwen3.5-2B configuration uses a text hidden width of 2,048 and 24 language layers. The feed-forward intermediate width is 6,144. Full-attention layers use 8 query heads and 2 key/value heads; the hybrid stack also defines 16 linear key heads and 16 linear value heads for the Gated DeltaNet path.

The vocabulary is very large, roughly a quarter-million token IDs. A large vocabulary changes where model capacity sits. Token embeddings and the output projection contain many parameters, while the 24 repeated blocks carry the contextual computation. Qwen ties the input and output embeddings, so one learned token representation participates at both ends of the language model.

The full-attention layer with grouped-query attention can be sketched head by head. If hidden state width is $d=2048$, each token first produces several query vectors, but fewer independent key/value groups:

$$
Q_h = XW_{Q,h},\qquad
K_g = XW_{K,g},\qquad
V_g = XW_{V,g}.
$$

Several query heads can share one $K_g,V_g$ pair. The query-head output is

$$
H_h=
\operatorname{softmax}\left(
\frac{Q_hK_{g(h)}^\top}{\sqrt{d_h}}+M
\right)V_{g(h)}.
$$

All head outputs are concatenated and projected through $W_O$.

For local inference, sharing K/V groups reduces the memory that grows with sequence length. If every query head required its own cached key and value vector at every position, the K/V cache would be much larger.

Qwen's published configuration also uses only part of each full-attention head for RoPE. The important learning point isn't the exact implementation constant; it is that content dimensions and position-rotated dimensions coexist inside the head. We don't alter that architectural choice in LoRA.

### What LoRA can and cannot change here

Our adapter targets the projection matrices inside every applicable language block. It doesn't add new transformer layers, change hidden width, create a new tokenizer, or change the full-attention/Gated-DeltaNet schedule.

If the base model learned a broad semantic representation in its 2,048-dimensional residual stream, LoRA can steer how information is routed and transformed inside that representation. It cannot expand the residual stream to 4,096 dimensions or replace a Gated DeltaNet layer with full attention.

This helps us set realistic expectations. Parameter-efficient fine-tuning is good at changing **behavioral policy** on top of a capable base. It is a poor method for teaching a tiny base a completely missing architecture or an enormous body of new factual knowledge. For factual freshness, Project 24 uses retrieval and point-in-time structured contexts instead of trying to bake daily market data into adapter weights.

### 1.10 Why the hybrid architecture helps long financial prompts

For ordinary full attention over $T$ tokens, the score matrix has $T^2$ query-key interactions per head. Ignoring implementation optimizations,

$$
\text{attention score work}\propto T^2d_h.
$$

At $T=8{,}192$, the number of pairwise positions is already about 67 million before multiplying by heads or layers. At much longer contexts, quadratic scaling becomes a central cost.

Autoregressive inference changes the shape of the problem. Once the first $T$ prompt tokens are processed, the model generates one new token at a time. With a K/V cache, the new query attends to cached keys/values rather than recomputing every old projection. The cache still grows with $T$, and each new full-attention token compares against a longer history.

A recurrent linear-attention memory aims for another scaling regime. Instead of keeping a separate key/value pair for every old token, it updates a fixed-size state. The per-token memory update can remain roughly constant with history length.

The tradeoff is representational. Full attention can directly say “the exact phrase I need is at token 6,204” and retrieve it by content similarity. A compressed recurrent state has to summarize history. Information can interfere or be forgotten.

Qwen3.5's 3:1 pattern gives most layers the efficient stateful mechanism while periodically inserting full attention. For our use case, that is attractive:

- evidence packets can be long;
- numbers and citation IDs may occur far before the answer;
- local hardware is limited;
- the response still needs exact retrieval behavior.

We shouldn't assume hybrid attention solves retrieval perfectly. Project 24 still spends considerable engineering effort on **evidence selection before prompting**. A good retrieval layer reduces the burden on the LLM by placing only relevant source passages and structured contexts in its window.

The architecture and the application therefore solve different bottlenecks:

$$
\text{RAG reduces irrelevant context}
\quad+\quad
\text{hybrid LLM processes the packed context efficiently}.
$$

If retrieval selects the wrong SEC section, no attention mechanism can recover the missing section. If retrieval selects the right passage but the LLM ignores it, model behavior becomes the problem.

### 1.11 Pretraining, instruction tuning and our LoRA stage are different learning problems

Large language models usually pass through several conceptual stages.

**Pretraining** learns next-token prediction over an enormous general corpus. It creates most of the language competence, representations and broad knowledge:

$$
\min_\theta
E_{x\sim\mathcal D_{\text{pretrain}}}
\left[
-\sum_t\log p_\theta(x_t\mid x_{<t})
\right].
$$

**Instruction/chat tuning** takes a pretrained model and makes conversation roles, instruction following and assistant-style responses more reliable.

Our **financial LoRA SFT** is narrower again. We start from an already capable Qwen3.5-2B model and optimize a small adapter on reviewed analyst examples.

The mathematical form of the loss is still next-token cross-entropy. What changes is the data distribution and which parameters are allowed to move.

That distinction explains why a relatively small supervised dataset can be useful. We are not asking thirty or a few hundred financial examples to teach the network English, arithmetic, JSON braces and the concept of inflation from scratch. Those capabilities are already represented in the base. We are reinforcing a response pattern around supplied evidence.

It also explains why the adapter shouldn't be treated as a live financial database. Suppose NVDA files a new 10-Q after training. Updating LoRA weights every quarter would be a slow and lossy way to add one filing. Project 24 instead retrieves the filing at query time.

A clean separation is:

| Layer | Changes slowly? | Stores/does |
|---|---|---|
| Base Qwen weights | very slowly | language/model capacity and broad pretrained knowledge |
| LoRA adapter | occasionally | analyst response policy and evidence discipline |
| Retrieval/document store | continuously | dated source text |
| Structured finance contexts | continuously | computed market/fundamental/macro features |
| User question | every call | current analytical objective |

This architecture makes the analyst auditable. We can update evidence without retraining, retrain behavior without rewriting source history, and upgrade the base model while keeping the application logic conceptually separate.

## 2. The Reproducible Training Recipe

The first code cell defines the entire run identity: data directories, the exact base model revision, the pinned `llama.cpp` revision, context and generation budgets, seed, optimizer settings, LoRA rank and target modules.

That may look like configuration plumbing, but in model training these values define the experiment. Saying “Qwen3.5-2B with LoRA” is still underspecified. A rank-8 adapter trained on a different data split with another learning rate is a different model. So is the same recipe against a later base-model revision.

The key settings are:

| Setting | Chosen value | Training meaning |
|---|---:|---|
| Base | Qwen3.5-2B, pinned revision | fixed pretrained starting point |
| Training context | 8,192 tokens | maximum complete training sequence |
| Epochs | 2 | at most two passes over accepted training examples |
| Learning rate | $5\times10^{-5}$ | LoRA update scale before optimizer adaptation |
| Micro-batch | 1 | one sequence fits on the GPU at a time |
| Gradient accumulation | 8 | eight micro-batches contribute to one optimizer step |
| LoRA rank | 16 | rank of each learned weight update |
| LoRA alpha | 16 | scaling parameter; $\alpha/r=1$ here |
| LoRA dropout | 0 | no stochastic dropout inside adapters |
| Evaluation interval | 50 update steps | validation-loss checkpoint cadence |
| Early-stop patience | 2 evaluations | stop after sustained lack of improvement |

With micro-batch 1 and accumulation 8, the nominal effective batch is

$$
B_{\text{effective}}=1\times8=8
$$

training sequences per optimizer update, except possibly the final incomplete accumulation window.

The seed is fixed to 3407. A seed can't guarantee bitwise equality across every CUDA kernel and library build, but it removes a major source of avoidable randomness from shuffling, initialization and generation.

The cell also uses persistent run and cache directories. On Colab, that can point into Drive. Large pretrained weights don't need to be downloaded every session, and a training run can resume from a full checkpoint rather than pretending a disconnected session is a new experiment.

### Why revisions are hashes rather than model names

`Qwen/Qwen3.5-2B` is a repository name whose files can evolve. A commit revision identifies one immutable snapshot. The same logic is used for `llama.cpp`: the converter and runtime are part of the deployed artifact, so their code revision belongs in provenance.

Once this cell runs successfully, we expect it to print or establish the resolved workspace locations and fixed hyperparameters. It doesn't train anything yet. It defines the recipe that later cells will either reproduce exactly or reject if a prior run used a different identity.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
from urllib.request import urlopen

on_colab = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
use_drive = True
drive_project = Path("/content/drive/MyDrive/quantfinlab_training")
if on_colab:
    if use_drive:
        from google.colab import drive
        drive.mount("/content/drive")
    training_home = drive_project if use_drive else Path("/content/quantfinlab_training")
    model_dir = Path("/content/quantfinlab-models")
    model_dir.mkdir(exist_ok=True)
    for name in ["requirements-training.in", "requirements-training.lock", "requirements-export.in", "requirements-export.lock"]:
        bundled = training_home / name
        destination = model_dir / name
        if bundled.exists():
            destination.write_bytes(bundled.read_bytes())
        elif not destination.exists():
            url = "https://raw.githubusercontent.com/ramtin-asadi/Quantitative-Finance-Lab/main/models/" + name
            with urlopen(url, timeout=60) as response:
                destination.write_bytes(response.read())
    data_dir = training_home / "data"
    run_dir = training_home / "runs/qwen-financial-lora"
    cache_dir = training_home / "cache/huggingface"
else:
    model_dir = Path.cwd() if Path("requirements-training.in").exists() else Path.cwd() / "models"
    model_dir = model_dir.resolve()
    data_dir = model_dir / "data" if (model_dir / "data/manifest.json").exists() else model_dir.parent / "workspace/financial_analyst/training/frozen"
    run_dir = model_dir / "outputs/qwen-financial-lora"
    cache_dir = model_dir / "local/huggingface"
base_model = "Qwen/Qwen3.5-2B"
base_revision = "15852e8c16360a2fea060d615a32b45270f8a8fc"
llama_revision = "3057bb66c86c46d5781e50e85462a760ba7d1feb"
training_context = 8192
generation_tokens = 1536
generation_seconds = 240
seed = 3407
epochs = 2
learning_rate = 5e-5
batch_size = 1
gradient_accumulation = 8
evaluation_steps = 50
early_stopping_patience = 2
acceptance_per_task = 6
validate_gguf = True
lora_config = {"rank": 16, "alpha": 16, "dropout": 0.0, "bias": "none",
               "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}
os.environ["HF_HOME"] = str(cache_dir)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_dir.mkdir(parents=True, exist_ok=True)
print("Training data:", data_dir, "\nPersistent output:", run_dir, "\nBase-model cache:", cache_dir)

### 2.1 What happens when the configuration cell runs

No gradient is computed. The cell determines whether it is running in Colab, optionally mounts Google Drive, finds the repository/model directories and creates cache/run locations. It then records every constant later cells need.

If a path is wrong, this is where we want the failure. A silent fallback to a temporary directory would make checkpoints disappear between sessions and could cause the adapter, merged model and exported GGUF to come from different runs.

A clean execution therefore establishes one answer to three questions:

1. **Which pretrained weights are we adapting?**
2. **Which frozen dataset and hyperparameters define this run?**
3. **Where will every checkpoint and exported artifact be written?**

The model is still untouched after this stage.

### 2.2 Pinned Software Environment

The next cell checks Linux and Python 3.12/3.13, reads the training requirements and installs the exact versions when necessary. Deep-learning stacks combine PyTorch, CUDA kernels, Transformers, PEFT, Unsloth, tokenizers and many transitive libraries. Small version changes can change APIs, precision behavior, serialization or generated text.

We therefore treat the package set like data provenance. Reproducibility is stronger when we can state:

$$
\text{result} =
f(\text{weights},\text{data},\text{code},\text{packages},\text{seed},\text{hardware}).
$$

A model hash alone identifies only the weights. It doesn't identify the software that interpreted those weights.

If the requirements are missing, the cell installs them and deliberately asks for a kernel restart. Reloading compiled libraries inside an already-running Python process can leave stale modules in memory. Restarting gives the pinned stack a clean process.

The cell also validates dependency compatibility rather than accepting a successful `pip` exit code as sufficient. If package A pins a range that conflicts with package B, we want to stop before allocating a 2B model and discovering the problem deep into training.

In [ ]:
import importlib.metadata
import subprocess

if sys.platform != "linux" or sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.12 or 3.13 on Linux, Colab, or WSL2. See models/README.md for setup.")
requirements = model_dir / "requirements-training.in"
missing = []
for spec in requirements.read_text(encoding="utf-8").splitlines():
    if not spec.strip() or spec.startswith("--"):
        continue
    name, expected = spec.split("==")
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(spec)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
    raise RuntimeError("Dependencies installed. Restart this notebook's kernel, then run from the first cell.")
from packaging.requirements import Requirement

for spec in requirements.read_text(encoding="utf-8").splitlines():
    if "==" not in spec:
        continue
    package = Requirement(spec)
    for dependency in importlib.metadata.requires(package.name) or []:
        dependency = Requirement(dependency)
        if dependency.marker and not dependency.marker.evaluate():
            continue
        if importlib.metadata.version(dependency.name) not in dependency.specifier:
            raise RuntimeError(f"Training dependency conflict: {package.name} requires {dependency}")
print("Pinned training environment is ready.")

### 2.3 Expected result of the environment check

On a prepared runtime, the cell should finish quickly and report that the pinned dependencies are already compatible. On a fresh runtime, installation is expected and the next correct action is a kernel restart followed by rerunning from the top.

This is one of the places where “automatic recovery” would be dangerous. A notebook that silently upgrades packages until imports succeed could produce a different training environment from the one recorded in the run recipe.

### 2.4 Precision, Memory and Hardware

The hardware gate comes before loading the model. This training recipe requires CUDA and checks for roughly 15 GB of GPU VRAM. It chooses BF16 when the GPU supports it and FP16 otherwise.

Both are 16-bit floating-point formats, but their exponent and mantissa allocations differ. FP16 has more fraction precision but a much narrower exponent range. BF16 uses the same 8-bit exponent width as FP32 and is therefore much harder to overflow in deep networks. When hardware supports BF16 well, it is usually a safer training format.

The important line is:

`load_in_4bit=False`.

We are **not doing QLoRA** here. QLoRA normally keeps a quantized base model in low-bit form during fine-tuning and learns LoRA adapters around that quantized representation. Our base parameters are loaded in BF16 or FP16. The LoRA parameters are subsequently forced to float32. Only after training and adapter merging do we quantize a deployment copy to Q4_K_M.

That gives three separate precision stages:

1. **pretrained base during adaptation:** BF16 or FP16;
2. **trainable LoRA matrices:** FP32;
3. **final local deployment artifact:** Q4_K_M GGUF after merging.

Quantization in stage 3 can reduce local memory and improve feasibility on a small machine without changing the optimization geometry used in stage 1.

### Gradient checkpointing

An LLM forward pass creates many intermediate activations. Backpropagation normally stores them so derivatives can be computed later. Activation memory grows with depth, sequence length and batch size.

Gradient checkpointing trades compute for memory. We store selected checkpoints and recompute intermediate activations during backward:

$$
\text{less activation memory}
\quad\Longleftrightarrow\quad
\text{extra forward computation}.
$$

At an 8,192-token context, that trade is valuable. It doesn't alter the objective; it changes how the same derivatives are obtained.

When this cell runs, we expect to see the GPU name, VRAM, selected dtype and a statement that the training base is 16-bit rather than 4-bit. If CUDA is missing or memory is too small, we stop here instead of failing halfway through model loading.

#### Training hardware and deployment hardware solve different problems

The training GPU threshold here shouldn't be carried into Project 24 as a deployment requirement. During fine-tuning we need room for long-sequence activations, backward gradients, LoRA optimizer state and checkpointed recomputation. After export, those training states disappear. The local application can therefore use a much smaller GPU for partial layer offload while system RAM stores the rest of the quantized model and context state.

This separation is one reason we do expensive adaptation once, then package the result for repeated local inference.

### 2.5 What the hardware check tells us before any training

A successful result says the runtime is capable of executing the recipe that follows; it is not evidence that training will converge. We still haven't loaded the adapter or computed loss.

The 15 GB threshold also explains why the later Project 24 notebook can run on a far smaller GPU than this training notebook. Training holds activations, gradients, optimizer state and adapter parameters. Local inference only needs the quantized merged weights, runtime buffers and K/V state. Training and inference therefore have very different memory economics.

In [ ]:
from unsloth import FastVisionModel

import gc
import hashlib
import json
import re
import shutil
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import Markdown, display
from transformers import (AutoTokenizer, DataCollatorForSeq2Seq, EarlyStoppingCallback,
                          StoppingCriteria, StoppingCriteriaList, Trainer, TrainerCallback,
                          TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint

if torch.__version__.split("+")[0] != "2.8.0":
    raise RuntimeError("Restart the kernel to load the installed PyTorch version before training.")
if not torch.cuda.is_available():
    raise RuntimeError("Training needs an NVIDIA CUDA GPU. CPU-only conversion can be run separately on saved merged weights.")
gpu = torch.cuda.get_device_properties(0)
bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if bf16 else torch.float16
if gpu.total_memory < 14 * 2**30:
    raise RuntimeError("This 16-bit LoRA recipe needs at least 15 GB VRAM. It does not load a 4-bit training base.")
if shutil.disk_usage(run_dir).free < 25 * 2**30:
    raise RuntimeError("Keep at least 25 GiB free for the cached base, checkpoints, merged weights and GGUF files.")
set_seed(seed)
display(pd.Series({"gpu": gpu.name, "vram_gib": gpu.total_memory / 2**30,
                   "base_precision": str(dtype), "adapter_precision": "float32"}))

### 2.6 Logging, Hashing and Atomic Metadata

The next utilities establish three habits used through the rest of the pipeline:

- timestamped logs tell us when long stages started and finished;
- SHA-256 hashes identify exact file contents;
- atomic JSON writes avoid half-written receipts if a run is interrupted.

For a file represented by bytes $b$, a cryptographic hash creates a deterministic digest

$$
h = \operatorname{SHA256}(b).
$$

The point isn't encryption. If one byte of a dataset, adapter or GGUF changes, its hash almost certainly changes. We can therefore use the digest as a content identity.

Atomic writes follow a simple transaction idea: write a complete temporary file and then rename it into the final location. The rename is much less likely to leave a valid-looking but truncated metadata file after a crash.

The helper that runs subprocesses also writes their stdout/stderr to export logs. Conversion and compilation happen outside Python later, so these logs become part of the model's provenance rather than disappearing in notebook scrollback.

In [ ]:
def log(message, **fields):
    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    print(f"[{stamp}] {message}" + (" | " + ", ".join(f"{key}={value}" for key, value in fields.items()) if fields else ""), flush=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)

def run_visible(command):
    log("Running", command=" ".join(map(str, command)))
    with (run_dir / "export.log").open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f"Command failed with exit code {returncode}; see {run_dir / 'export.log'}.")

### 2.7 Expected behavior of the utility cell

This stage should be quiet: it defines tools and may print a small status line, but it doesn't change model weights. Its value appears later when a run can prove which dataset produced an adapter, which adapter produced a merged model, and which merged model produced the quantized GGUF.

If a later receipt says an artifact with hash $h$ was already built from exactly the same identity, the pipeline can reuse it. If any identity field changes, reuse is rejected.

## 3. Evidence-Grounded Supervision

Before teaching the language model anything, we validate the frozen train/validation files against their manifest. Each training example is more than a question-answer pair. It carries:

- an example ID and grouping identity;
- a task family;
- a cutoff timestamp;
- source/evidence IDs;
- source hashes;
- evidence text with its own availability time;
- a reviewed assistant completion.

The status has to be `frozen`, and file hashes must match the manifest. This prevents a subtle but serious reproducibility problem: changing one example after training while keeping the same run label.

### Time is part of supervision

Financial analysis is unusually sensitive to future information. If an example asks what could be known as of date $t$, every evidence item must satisfy

$$
\text{available\_at}_i \le t.
$$

The code checks that condition for every packet. It also requires strict chronology between the two splits:

$$
\max(t_{\text{train}}) < \min(t_{\text{validation}}).
$$

Randomly splitting neighboring financial examples can leak repeated events, filings or near-identical market states across train and validation. A chronological split asks a harder and more realistic question: can the adapted behavior generalize to later, unseen evidence?

### Overlap can occur at several levels

Checking only `example_id` would be weak. Two examples could have different IDs but cite the same source document or use the same source text. The validation therefore checks overlap in:

- example IDs;
- group IDs;
- source IDs;
- source hashes.

This is closer to how we approached temporal leakage in Projects 16 and 19: the split has to be independent at the level where information can actually leak.

### Task balance

The dataset spans five task families:

- **event** — explain a dated economic/market event from supplied evidence;
- **SEC change** — compare filing evidence and identify material changes;
- **macro** — interpret economic-release evidence;
- **reconciliation** — handle evidence that has to be reconciled rather than copied blindly;
- **market** — synthesize market contexts across assets.

We display task counts and cutoff distributions before training. If one task dominates the dataset, aggregate validation loss could improve while a smaller task deteriorates. The task mix is therefore part of what we need to understand when reading later acceptance checks.

### 3.1 Supervised examples teach a decision policy through contrasts in evidence

A useful financial fine-tuning example should not only contain correct prose. It should teach the model what evidence deserves weight and what uncertainty remains.

Consider two hypothetical packets.

**Packet A**
- revenue +20%;
- operating income +25%;
- operating cash flow +22%;
- no material leverage increase.

**Packet B**
- revenue +20%;
- operating income +25%;
- operating cash flow -35%;
- receivables surge.

If both target answers simply say “strong growth,” the dataset teaches the model to copy headline earnings and ignore cash conversion. If Packet B's target instead says “reported profitability is strong, but cash conversion weakened and deserves investigation,” the supervision contains a meaningful contrast.

The same principle applies to market and macro examples:

- equity rally + tighter credit + falling volatility is a coherent risk-on cluster;
- equity rally + weak breadth + rising front-end yields + commodity weakness is mixed;
- headline CPI down because energy falls while core services stay firm is different from broad disinflation;
- a new SEC risk-factor sentence is different from a mechanically repeated boilerplate section.

The quality of supervised fine-tuning therefore depends on **analytical labels**, not only formatting.

### Time-sliced validation tests a realistic failure mode

Financial distributions change. Later examples can contain new rate regimes, market stress, accounting patterns or event combinations. Chronological validation makes generalization harder than random iid splitting, but that is desirable.

If the model memorizes phrasing around an earlier CPI release, a random split might reward it for seeing nearly identical material on both sides. A later validation window forces the adapter to transfer the underlying evidence-use policy.

### Source hashes protect against hidden duplicates

Two source IDs can point to the same text if ingestion pipelines rename or replicate documents. Hashing source content catches that form of leakage.

This matters especially in SEC and release archives, where the same filing or release can be represented through several metadata paths. The validation rules are stricter than simple row deduplication because the semantic unit is the source evidence.

In [ ]:
manifest = json.loads((data_dir / "manifest.json").read_text(encoding="utf-8"))
if manifest["status"] != "frozen":
    raise ValueError("Use a reviewed, frozen train/validation split.")
records = {}
for split in ["train", "validation"]:
    path = data_dir / f"{split}.jsonl"
    if sha256(path) != manifest["files"][path.name]:
        raise ValueError(f"Dataset checksum differs from its manifest: {path.name}")
    records[split] = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(records[split]) != manifest["counts"][split]:
        raise ValueError(f"Unexpected number of {split} examples.")
    for row in records[split]:
        packet = json.loads(row["messages"][1]["content"])
        cutoff = pd.Timestamp(packet["as_of"])
        if row["quality_status"] != "accepted" or cutoff != pd.Timestamp(row["cutoff"]):
            raise ValueError(f"Unreviewed example or inconsistent cutoff: {row['example_id']}")
        for evidence in packet["evidence"]:
            if pd.Timestamp(evidence["available_at"]) > cutoff or hashlib.sha256(evidence["text"].encode()).hexdigest() != evidence["text_hash"]:
                raise ValueError(f"Invalid evidence date or text hash: {row['example_id']}")
for field in ["example_id", "group_id", "source_ids", "source_hashes"]:
    inventories = []
    for split in ["train", "validation"]:
        values = set()
        for row in records[split]:
            value = row[field]
            values.update(value.values() if isinstance(value, dict) else value if isinstance(value, list) else [value])
        inventories.append(values)
    if inventories[0] & inventories[1]:
        raise ValueError(f"Training/validation leakage in {field}.")
train_end = max(pd.Timestamp(row["cutoff"]) for row in records["train"])
validation_start = min(pd.Timestamp(row["cutoff"]) for row in records["validation"])
if train_end >= validation_start:
    raise ValueError("Validation must be strictly later than training.")
task_counts = pd.DataFrame({split: pd.Series([row["task"] for row in rows]).value_counts()
                            for split, rows in records.items()})
display(task_counts)
task_counts.plot.bar(title="Training and later validation examples", ylabel="Examples", rot=20)
plt.show()
log("Dataset hashes and chronological separation passed", train_end=train_end, validation_start=validation_start)

### 3.2 What the frozen-data audit should show

A clean run should confirm:

- train and validation hashes equal the manifest;
- every record is reviewed and accepted;
- no evidence arrives after its question cutoff;
- no prohibited identity/source overlap crosses the split;
- the latest training cutoff is earlier than the earliest validation cutoff.

The count table and date plot are descriptive. They let us see whether the data are concentrated in one task or one narrow time period.

A failure here should stop the experiment. Training on a temporally contaminated dataset and then “noting the limitation” after the fact would make the validation metrics difficult to interpret.

### 3.3 Structured Financial Targets and Validators

The next cell defines the assistant output with Pydantic models. Each answer contains:

- a concise conclusion;
- a materiality category;
- claims;
- what changed;
- why it matters;
- uncertainty.

Each claim has a statement, one or more evidence IDs, and a kind: `fact`, `interpretation`, or `uncertainty`.

This converts a vague instruction like “be grounded” into a testable contract. A fact can be checked against evidence. An interpretation can cite its support. A numerical token can be traced. Unknown fields are forbidden.

### Numerical traceability

Suppose the evidence says revenue was `$35.08 billion` and an answer claims `$53.08 billion`. Both strings are syntactically valid numbers. Ordinary JSON-schema validation would accept the response.

The training/evaluation utilities therefore extract numerical tokens with decimal arithmetic and ask whether each number in a claim or summary can be traced to its cited evidence. Conceptually,

$$
\mathcal N(\text{claim})
\subseteq
\bigcup_{e\in C(\text{claim})}\mathcal N(e),
$$

where $\mathcal N(\cdot)$ is the set of normalized numbers and $C(\text{claim})$ is the cited evidence set.

This is intentionally conservative. It discourages the model from inventing arithmetic unless the resulting number is already supported by the evidence. Project 24 later relies on precomputed structured contexts for derived financial metrics, so the model can cite a ratio that our finance code calculated instead of freehanding the calculation in natural language.

### Dynamic evidence enums

For a particular packet, valid evidence IDs are known. The JSON schema can therefore restrict citations to exactly those IDs. If the prompt contains `e1`, `e2`, and `e3`, the model can't legally cite `e9`.

That is stronger than asking in prose to “cite your sources.” We move an important behavior from natural-language compliance into constrained decoding and post-generation validation.

### Duplicate and reference checks

The code also rejects duplicate claims and validates target references. Repetition is a common small-model failure mode: a model can consume output tokens by restating the same sentence in several fields. Duplicate detection makes that behavior visible.

For evaluation, we select six validation examples per task, giving thirty adapter-acceptance cases. After quantization we use one case per task to verify that the final GGUF still obeys the same contract.

### 3.4 Evidence grounding as a constrained prediction problem

Standard supervised fine-tuning asks the model to imitate target text. Our examples add an explicit evidence graph.

Let the packet contain evidence objects

$$
E=\{e_1,\ldots,e_m\}.
$$

The answer contains claims

$$
C=\{c_1,\ldots,c_n\},
$$

and each claim carries a citation set

$$
R(c_i)\subseteq E.
$$

For a factual claim, we want three properties:

1. **referential validity:** every cited ID exists;
2. **text/numeric support:** the cited evidence contains the factual content we claim;
3. **temporal validity:** cited evidence was available by the example cutoff.

We cannot fully prove semantic entailment with simple deterministic code. We therefore use strong checks that are cheap and auditable: ID membership, numerical traceability, source hashes, time cutoffs and duplicate controls.

This design has an important consequence for how we read automatic pass rates. Passing means “the output obeyed these explicit validators.” It doesn't mean “a financial analyst agrees with every inference.”

That boundary is healthy. A deterministic validator should be narrow enough that we understand its false positives and false negatives. Asking another LLM to judge everything would create a second opaque model whose errors are harder to audit.

### Why interpretations may cite evidence without copying it

A fact can often be traced almost verbatim:

> Operating cash flow fell 52.18%.

An interpretation transforms evidence:

> Cash conversion weakened relative to reported earnings.

The second sentence doesn't have to appear literally in the source. It should cite the factual inputs that justify it. We therefore allow `kind="interpretation"` as a separate claim type rather than forcing every analytical sentence into extractive quotation.

The price is that interpretation quality has to be reviewed separately. Project 24 later gives us examples where an interpretation cites correct evidence and still overstates the economic conclusion.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict
from pydantic import Field, ValidationError
from decimal import Decimal

class Claim(BaseModel):
    model_config = ConfigDict(extra="forbid")
    statement: str = Field(min_length=1)
    evidence_ids: list[str] = Field(min_length=1)
    kind: Literal["fact", "interpretation", "uncertainty"]

class Analysis(BaseModel):
    model_config = ConfigDict(extra="forbid")
    conclusion: str = Field(min_length=1)
    materiality: Literal["low", "medium", "high", "uncertain"]
    claims: list[Claim] = Field(min_length=1)
    what_changed: str = Field(min_length=1)
    why_it_matters: str = Field(min_length=1)
    uncertainty: list[str] = Field(min_length=1)

def numerical_values(text):
    return {str(Decimal(value.replace(",", "")).normalize())
            for value in re.findall(r"(?<![\w])[-+]?\d[\d,]*(?:\.\d+)?", text)}

def check_output(text, row):
    try:
        result = Analysis.model_validate_json(text)
    except ValidationError as error:
        details = error.errors(include_url=False, include_input=False)
        return {"schema": False, "duplicate_claims": 0,
                "errors": [f"{item['loc']}: {item['msg']}" for item in details]}
    statements = [" ".join(claim.statement.casefold().split()) for claim in result.claims]
    duplicate_claims = len(statements) - len(set(statements))
    errors = ["Repeated claim statements"] if duplicate_claims else []
    evidence = {item["evidence_id"]: item for item in json.loads(row["messages"][1]["content"])["evidence"]}
    for i, claim in enumerate(result.claims):
        if set(claim.evidence_ids) - evidence.keys():
            errors.append(f"Claim {i}: unknown evidence ID")
            continue
        source = " ".join(evidence[key]["text"] for key in claim.evidence_ids)
        missing = numerical_values(claim.statement) - numerical_values(source)
        if missing:
            errors.append(f"Claim {i}: untraceable numbers {sorted(missing)}")
    prose = " ".join([result.conclusion, result.what_changed, result.why_it_matters, *result.uncertainty])
    missing = numerical_values(prose) - numerical_values(" ".join(e["text"] for e in evidence.values()))
    if missing:
        errors.append(f"Summary: untraceable numbers {sorted(missing)}")
    return {"schema": True, "duplicate_claims": duplicate_claims, "errors": errors}

def output_schema(row):
    schema = Analysis.model_json_schema()
    evidence = json.loads(row["messages"][1]["content"])["evidence"]
    schema["$defs"]["Claim"]["properties"]["evidence_ids"]["items"] = {
        "type": "string", "enum": [item["evidence_id"] for item in evidence]}
    return schema

for row in records["train"] + records["validation"]:
    result = check_output(row["messages"][-1]["content"], row)
    if not result["schema"] or result["errors"]:
        raise ValueError(f"Reference target failed validation: {row['example_id']}: {result['errors']}")


tasks = ["event", "sec_change", "macro", "reconciliation", "market"]
acceptance_rows = []
for task in tasks:
    group = sorted([row for row in records["validation"] if row["task"] == task], key=lambda row: row["example_id"])
    acceptance_rows.extend(group[:acceptance_per_task])
assert len(acceptance_rows) == len(tasks) * acceptance_per_task
print("Reference checks passed. Adapter checks:", len(acceptance_rows), "| GGUF checks:", len(tasks) if validate_gguf else 0)

### 3.5 What the schema cell prepares

This cell still doesn't alter the model. It creates the definitions used by:

- training-example validation;
- constrained generation;
- adapter acceptance;
- post-quantization acceptance;
- the later Project 24 runtime.

A clean execution tells us the task list and acceptance sample have been constructed successfully. The strongest point is that the same behavioral contract follows the model all the way from supervised examples to the quantized local deployment artifact.

## 4. From Reviewed Conversations to Causal-LM Training Examples

We now convert reviewed conversations into the exact token sequences the model will train on. This is where ordinary text becomes a causal-language-model dataset.

The tokenizer is loaded from the **same pinned base revision** as the weights. Padding is on the right, and if Qwen doesn't expose a separate padding token we reuse EOS. The chat template is hashed and later stored with the artifact.

A chat model doesn't receive Python dictionaries such as

```text
{"role": "user", "content": "..."}
```

directly. The template serializes roles and text into the model's learned conversation format. For Qwen, role-boundary tokens tell the model where system, user and assistant turns begin and end. The exact template therefore affects the token sequence and belongs in the reproducibility identity.

### Prompt and completion are one causal sequence

Take a simplified example:

```text
system: Use only supplied evidence.
user: What changed in the CPI release?
assistant: {"conclusion": ...}
```

The causal model sees one serialized stream. If prompt length is $P$ and the supervised answer has $A$ tokens, the full sequence length is roughly

$$
T=P+A+1,
$$

where the final token accounts for EOS.

The input IDs contain all $T$ tokens. The labels are different:

$$
y_t =
\begin{cases}
-100, & t\le P,\\
x_t, & P<t\le T.
\end{cases}
$$

Hugging Face's cross-entropy ignores label `-100`. So the objective is

$$
\mathcal L
=
-\sum_{t=P+1}^{T}
\log p_\theta(x_t\mid x_{<t}).
$$

We call this **completion-only supervised fine-tuning**.

Why mask the prompt? The goal is to teach the assistant response conditional on evidence, not to spend capacity learning to regenerate system instructions, user questions, or retrieved documents. The prompt is still visible through causal attention. It simply doesn't generate direct loss.

### Teacher forcing

During supervised training, the correct previous answer tokens are present in the prefix when predicting the next one. If the gold completion is

`{"materiality":"high"...}`

the model predicts each next token while seeing the correct earlier gold tokens. This is **teacher forcing**.

Inference is different. The model sees its own generated history. An early malformed quote or brace can push later probabilities into a bad region. That training/inference mismatch is one reason we later combine:

- constrained JSON schema;
- deterministic decoding;
- post-generation validators;
- one evidence-based repair attempt.

Fine-tuning alone doesn't guarantee syntactically perfect long outputs.

### No silent truncation

The training context is 8,192 tokens. A common pipeline would truncate longer examples. We reject them instead.

Truncation would be dangerous here. If the tail of the assistant answer disappeared, the model might train on incomplete JSON. If the beginning of the evidence disappeared, a cited answer could lose the source that supports it. If citation IDs were cut but claims remained, supervision would become internally inconsistent.

The rule is therefore:

$$
T \le 8192
$$

for every encoded example, otherwise the dataset has to be fixed deliberately.

### Thinking is disabled

The chat template is rendered with `enable_thinking=False`. The training target is the final analyst structure, not hidden or verbose reasoning text. The code even verifies the exact assistant prefix expected from the pinned Qwen template. That protects against a template/version change silently placing the loss mask at the wrong boundary.

Once this cell runs, we get encoded train and validation datasets plus token-length distributions. Those distributions are worth reading: they show how much of the 8,192-token budget the examples actually use and whether a few unusually long packets dominate compute.

### 4.1 Token boundaries, numbers and source IDs are part of the learning problem

Financial text contains patterns that are unusually sensitive to tokenization:

- decimals such as `0.70`;
- percentages such as `52.18%`;
- basis points;
- dates;
- long SEC evidence IDs;
- JSON punctuation.

A human sees `52.18 percent` as one quantity. The tokenizer may represent the digits, decimal point and suffix with several tokens. The loss is applied token by token.

That gives numerical generation an important property: there is no built-in numeric register that copies one floating-point value from evidence to answer. The model predicts token pieces. It can generate `52.18` correctly, but it can also produce `52.8`, transpose digits or interpolate a plausible value.

The deterministic numerical-traceability validator compensates for that weakness. We don't ask a 2B language model to be the authoritative arithmetic engine. Structured finance code computes ratios; evidence packets contain the resulting numbers; the answer is checked against those numbers.

### Long source IDs also benefit from constrained generation

An evidence ID such as

```text
context-fundamentals-554c2b7c75527632
```

is semantically arbitrary. The model should not have to “understand” the hash suffix. It only has to copy/select a valid identifier.

If we leave citation generation unconstrained, one token error can create a nonexistent ID. A dynamic schema that enumerates packet IDs turns citation selection into a constrained choice.

This is a general system-design lesson: don't spend model capacity learning rules that deterministic software can enforce exactly.

### Chat-role boundaries protect the conditional structure

The serialized prompt contains system instructions, user text and evidence. Role tokens tell the chat model how to interpret each region.

The training labels begin only after the assistant prefix. If we accidentally included user/evidence tokens in the supervised loss, the adapter would be rewarded for predicting the prompt itself. That can encourage copying and waste gradient.

If we accidentally started supervision *after* part of the assistant answer, early output structure would remain entirely dependent on the base model.

The explicit prefix assertion in the code is therefore one of the most important low-level correctness checks in the training pipeline.

#### Numerical reasoning in token space needs an external audit

Financial prompts are full of objects that look simple to us but are awkward for a language model: percentages, basis points, signed changes, ratios, dates, source IDs and units.

A transformer doesn't receive the number $4.63\%$ as a floating-point scalar with a known economic type. The tokenizer turns the character sequence into one or more token IDs. The model then predicts subsequent token IDs from learned statistical structure. Depending on the tokenizer, strings such as `4.63`, `4.64`, `463 bp` and `0.0463` can have very different token decompositions even though they describe closely related numerical quantities.

This creates several possible failure modes.

**Digit substitution.**  
The model can generate a nearby-looking value because the language objective rewards probable token sequences, not exact arithmetic identity.

**Unit substitution.**  
A move of 7 basis points can become 7 percent if the answer loses the unit carried by the evidence.

**Sign errors.**  
A negative spread change can be described as widening if the model focuses on the magnitude and loses the relation between sign and financial meaning.

**Derived-number invention.**  
The answer can calculate a new percentage or difference that wasn't supplied and make a small arithmetic error.

**Source-number mismatch.**  
A correct number can be attached to the wrong evidence ID.

Fine-tuning can reduce these errors by repeatedly showing the desired behavior, but the causal-LM objective still predicts text. We therefore give exact numerical work to deterministic code whenever possible and validate generated numbers against the evidence packet.

Suppose the evidence contains:

```text
E1: 2Y yield = 4.63 percent; 1d change = +7 bp
E2: 10Y yield = 4.96 percent; 1d change = +1 bp
```

The model can safely interpret the pattern as stronger front-end repricing. If it repeats `4.63`, `+7`, `4.96` or `+1`, the validator can require those tokens to be traceable to eligible evidence. If we also need the change in the 2s10s spread, deterministic finance code can calculate it before prompting rather than asking the model to do basis-point arithmetic inside free generation.

This division of labor becomes a recurring design rule in Project 24:

- **Python calculates;**
- **retrieval finds;**
- **the LLM synthesizes;**
- **validators check what can be checked deterministically;**
- **an analyst still reviews economic meaning.**

That architecture is especially useful for a small local model. We don't need the model to become a calculator, database and source-of-truth system at the same time. We want it to become better at transforming supplied, dated evidence into disciplined financial language.

### 4.2 EOS is part of the target, not an afterthought

The encoded target appends the end-of-sequence token after the reviewed JSON.

That teaches a probability for stopping:

$$
p(\text{EOS}\mid x,\text{complete answer}).
$$

Without EOS supervision, a model can learn the answer body but have weak pressure to terminate. It may continue with another JSON object, commentary, or repeated fields.

At inference, reaching EOS before the token/time ceiling is one of the acceptance conditions. Training and deployment therefore agree on what “finished” means.

### Why right padding is a safe choice here

Right padding places padding tokens after real sequence content. The attention mask and `-100` labels prevent those padded positions from acting as supervised targets.

The causal order of actual prompt/answer tokens remains untouched. This is especially intuitive for decoder-only causal models because real content stays left-aligned.

### Token counts are also a cost measure

One example with 8,000 tokens requires far more attention/activation work than one with 1,000 tokens. The token-length plots therefore describe not only data shape but expected training cost.

When we later build Project 24 packets, we explicitly budget prompt tokens for the same reason. Token economics connects model training to application design.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model, revision=base_revision, cache_dir=str(cache_dir))
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
chat_template = tokenizer.chat_template
chat_template_hash = hashlib.sha256(chat_template.encode()).hexdigest()

def prompt_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def encode_example(row):
    prompt = prompt_text(row["messages"][:2])
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    completion_ids = tokenizer.encode(row["messages"][-1]["content"], add_special_tokens=False) + [tokenizer.eos_token_id]
    input_ids = prompt_ids + completion_ids
    if len(input_ids) > training_context:
        raise ValueError(f"Example exceeds the context; no silent truncation: {row['example_id']}")
    if not prompt.endswith("<|im_start|>assistant\n<think>\n\n</think>\n\n"):
        raise ValueError("Unexpected non-thinking assistant prefix; inspect the tokenizer template.")
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
            "labels": [-100] * len(prompt_ids) + completion_ids}

encoded = {split: Dataset.from_list([encode_example(row) for row in rows]) for split, rows in records.items()}
lengths = pd.DataFrame([{"split": split, "task": row["task"], "tokens": len(item["input_ids"]),
                        "answer_tokens": sum(token != -100 for token in item["labels"])}
                       for split, rows in records.items() for row, item in zip(rows, encoded[split])])
display(lengths.groupby("split")[["tokens", "answer_tokens"]].describe().round(1))
lengths.groupby("split").tokens.plot.hist(alpha=0.5, bins=30, legend=True)
plt.xlabel("Prompt plus supervised answer tokens")
plt.show()

### 4.3 Reading the token-length diagnostics when the cell is run

There is no saved output in the source notebook, so we shouldn't invent medians or maxima. The plot will show the empirical distribution from the frozen data.

We would look for three things:

1. **Sequences close to 8,192.** A large mass at the ceiling would suggest the training window is constraining the dataset design.
2. **A very long right tail.** With batch size one, long examples take much more compute and activation memory than short examples, so a few records can dominate wall-clock time.
3. **Train/validation mismatch.** If validation packets are systematically longer, evaluation loss may partly reflect a harder context-length regime rather than only chronological generalization.

The cell also verifies the assistant prefix and the exact tokenized answer boundary. If that check fails, training should stop. A loss mask shifted by one role marker would optimize the wrong text while still producing a perfectly finite loss.

### 4.4 Dynamic Padding and the Training Batch

The collator takes encoded examples and pads the shorter sequence in a batch to the longest sequence in that batch. Padding makes tensors rectangular:

$$
X\in\mathbb{N}^{B\times T_{\max}}.
$$

Input padding positions are masked from attention as appropriate, and label padding is set to `-100`, so those artificial tokens don't contribute to loss.

The collator pads to a multiple of eight. That can improve tensor-core alignment on modern GPUs while adding only a small number of padding positions.

With our micro-batch size of one, dynamic padding seems almost irrelevant during the actual training step: each micro-batch contains a single example. The cell deliberately collates two examples as a **verification test**. It then decodes only labels that are not `-100` and checks that they equal the reviewed assistant answer plus EOS.

This is a very useful unit test. A training loop can run for hours with the wrong labels and never throw an exception. Here we inspect the exact supervised token span before a single gradient update occurs.

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100,
                                pad_to_multiple_of=8, return_tensors="pt")
batch = collator([encoded["train"][0], encoded["train"][1]])
for row, original in zip(batch["labels"], [encoded["train"][0], encoded["train"][1]]):
    supervised = row[row.ne(-100)].tolist()
    expected = [token for token in original["labels"] if token != -100]
    assert supervised == expected and supervised[-1] == tokenizer.eos_token_id
    assert row[:len(original["labels"]) - len(expected)].eq(-100).all()
display(pd.DataFrame({"sequence_tokens": batch["attention_mask"].sum(1).tolist(),
                      "supervised_tokens": batch["labels"].ne(-100).sum(1).tolist()}))
display(Markdown("**Supervised answer, excluding EOS**\n\n" + tokenizer.decode(supervised[:-1])))
del batch
log("Completion-only masks, padding and EOS checked without a model forward pass")

### 4.5 What the collator test should establish

When run, the printed/decoded supervised region should contain only the target assistant completion and its EOS marker. We should not see:

- system instructions;
- user question text;
- retrieved evidence copied from the prompt;
- padding tokens.

If any of those appear, the completion-only objective is wrong.

This test also separates **sequence padding** from **gradient accumulation**. Padding shapes one micro-batch. Accumulation later combines gradients from eight separate forward/backward passes before one optimizer step.

### 4.6 Run Identity, Resume State and Reproducibility

Before loading the model, we build a reproducibility recipe containing:

- package versions;
- base model and commit revision;
- dataset hashes;
- chat-template hash;
- tokenizer/training settings;
- LoRA settings;
- trainer settings.

We hash or compare that recipe against any existing run directory.

Suppose yesterday's run used rank 16 and today's code changes rank to 32. Reusing yesterday's optimizer checkpoint just because the directory name is the same would mix two different parameterizations. The code refuses that state.

The maximum scheduled number of optimizer updates is approximately

$$
N_{\text{updates}}
=
\left\lceil
\frac{N_{\text{train}}}
{B_{\text{micro}}\times G}
\right\rceil
\times E,
$$

where $G=8$ accumulation steps and $E=2$ epochs. The actual number can be smaller if early stopping triggers.

A checkpoint here means more than adapter weights. To resume optimization faithfully we need:

- optimizer moments;
- learning-rate scheduler state;
- RNG state;
- mixed-precision scaler state where relevant;
- trainer progress;
- adapter parameters.

Restarting only from the adapter weights would be a new optimization path, not a continuation of the interrupted one.

### 4.7 Determinism has layers

Fixing a random seed is useful, but a reproducible training run has several levels.

**Data determinism**  
The same frozen JSONL bytes and same split.

**Preprocessing determinism**  
The same tokenizer revision, chat template and masking logic.

**Initialization determinism**  
The same base revision and LoRA initialization seed.

**Optimization determinism**  
The same batch order, hyperparameters, optimizer/scheduler state and checkpoint resume state.

**Kernel determinism**  
GPU kernels can still introduce small nondeterministic floating-point differences depending on the operation and platform.

We therefore don't promise that two machines must produce byte-identical final adapters merely because they share seed 3407. We do make the experiment specific enough to investigate differences.

The SHA-256 receipts are especially useful after the run. If adapter hashes differ, we know the artifacts are not identical even if their filenames are.

### Resume semantics and optimizer history

AdamW's state includes $m_t$ and $v_t$. If we load only the adapter weights $\theta_t$ but reset those moments to zero, the next update becomes

$$
\theta_{t+1}
=
\theta_t
-
\eta\cdot \operatorname{AdamStep}(g_{t+1};m_0=0,v_0=0),
$$

instead of using the historical optimizer state. That is a different training trajectory.

We therefore retain optimizer, scheduler, RNG and scaler state. “Resume” means resume the optimization process, not simply start a new run from the last adapter.

This also explains why we keep the final adapter separately. A deployment adapter needs only learned LoRA tensors and metadata; a resumable training checkpoint needs much more state.

### 4.8 Reuse, resume and “already complete” are different states

When the recipe matches, the code can distinguish:

- **no prior run:** start from the pinned base;
- **incomplete run:** resume the latest checkpoint;
- **completed run:** verify the saved adapter hash and reuse the completed artifact.

That distinction prevents a common notebook problem where rerunning cells accidentally trains a finished model for additional epochs.

The scheduled-update count printed here gives us a useful scale before training begins, but it isn't a performance metric. Validation behavior later determines whether the run is worth exporting.

In [ ]:
versions = {spec.split("==")[0]: importlib.metadata.version(spec.split("==")[0])
            for spec in requirements.read_text(encoding="utf-8").splitlines() if "==" in spec}
recipe = {"version": "qwen-financial-lora-1", "base_model": base_model, "base_revision": base_revision,
          "dataset_hashes": manifest["files"], "chat_template_hash": chat_template_hash, "lora": lora_config,
          "trainer": {"epochs": epochs, "learning_rate": learning_rate, "batch_size": batch_size,
                      "gradient_accumulation": gradient_accumulation, "training_context": training_context,
                      "dtype": str(dtype), "evaluation_steps": evaluation_steps,
                      "early_stopping_patience": early_stopping_patience, "seed": seed}, "packages": versions}
recipe_path = run_dir / "training_manifest.json"
if recipe_path.exists() and json.loads(recipe_path.read_text(encoding="utf-8")) != recipe:
    raise ValueError("This output folder belongs to another recipe. Choose a new run_dir to change the recipe.")
save_json(recipe_path, recipe)
checkpoints_dir = run_dir / "checkpoints"
checkpoints_dir.mkdir(exist_ok=True)
last_checkpoint = get_last_checkpoint(str(checkpoints_dir))
adapter_dir = run_dir / "adapter"
trained_path = run_dir / "trained.json"
trained = json.loads(trained_path.read_text(encoding="utf-8")) if trained_path.exists() else None
if trained and sha256(adapter_dir / "adapter_model.safetensors") != trained["adapter_sha256"]:
    raise ValueError("The completed adapter checksum changed.")
log("Training state", completed=trained is not None, resume_checkpoint=last_checkpoint)
log("Maximum scheduled updates", count=int(np.ceil(len(encoded["train"]) / (batch_size * gradient_accumulation)) * epochs))

## 5. LoRA Fine-Tuning of Qwen3.5-2B

We now reach the central training method.

A dense linear layer in the pretrained model maps

$$
y = Wx,
$$

with

$$
W\in\mathbb{R}^{d_{\text{out}}\times d_{\text{in}}}.
$$

Full fine-tuning would update every element of $W$. For a large model, doing that across all layers creates gradients and optimizer state for billions of parameters.

LoRA freezes $W$ and learns a low-rank update:

$$
W' = W + \Delta W,
$$

$$
\Delta W = \frac{\alpha}{r}BA,
$$

where

$$
A\in\mathbb{R}^{r\times d_{\text{in}}},
\qquad
B\in\mathbb{R}^{d_{\text{out}}\times r}.
$$

The rank $r$ is much smaller than the original dimensions. Here $r=16$ and $\alpha=16$, so the scalar multiplier is

$$
\frac{\alpha}{r}=1.
$$

For one matrix, full tuning would expose

$$
d_{\text{out}}d_{\text{in}}
$$

trainable values. LoRA exposes

$$
r(d_{\text{in}}+d_{\text{out}}).
$$

If both dimensions were 2,048, full tuning would mean about 4.19 million parameters for that matrix, while rank-16 LoRA would use

$$
16(2048+2048)=65{,}536.
$$

That example gives the geometry; the actual Qwen projections have several different shapes.

### Why a low-rank update can work

The assumption is not that the pretrained matrix itself is low rank. It is that the **task-specific change we need** can often be represented in a much smaller subspace than the full matrix.

Qwen already knows language, JSON syntax, financial vocabulary and a broad amount of public knowledge. We want to shift behavior toward:

- evidence-grounded conclusions;
- valid citation IDs;
- numerical traceability;
- concise materiality judgments;
- explicit uncertainty;
- the task structures in our reviewed examples.

That is a much narrower adaptation problem than learning language from scratch.

### Which Qwen matrices receive adapters

The target list covers:

**Attention projections**
- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`

**Feed-forward projections**
- `gate_proj`
- `up_proj`
- `down_proj`

Adding LoRA to Q/K/V can alter how the model queries, addresses and carries prompt information. Adapting `o_proj` changes how attended features return to the residual stream. Adapting the MLP projections changes nonlinear feature extraction after sequence mixing.

The adapters are enabled for the **language model only**. Visual layers are frozen. This is a text analyst, so training the vision encoder would consume memory and parameters without supervision.

### LoRA initialization and learning

A typical LoRA setup initializes the composite update near zero so the starting model behaves like the pretrained base. Gradient descent then updates $A$ and $B$ while $W$ stays fixed:

$$
\nabla_W\mathcal L = 0,
\qquad
\nabla_A\mathcal L\ne0,
\qquad
\nabla_B\mathcal L\ne0.
$$

The forward pass still uses the base computation. “Frozen” means no optimizer update; it doesn't mean the base layers disappear from the graph.

### This is 16-bit LoRA, not QLoRA

The loader explicitly sets `load_in_4bit=False`. The base is BF16/FP16. Trainable LoRA tensors are forced to float32.

QLoRA would keep the base quantized during training and reconstruct/dequantize values as needed for computation. That can cut memory further, but it introduces quantization into the adaptation path. Our recipe instead prioritizes a clean 16-bit base during optimization and uses quantization only for the final local inference copy.

### 5.1 Why fine-tuning a small financial analyst is mostly about behavior

It is useful to separate **capability**, **knowledge** and **policy**.

The pretrained Qwen model already has broad language capability. It can parse a sentence, produce JSON-like text, recognize many financial terms and follow ordinary instructions.

Its parametric knowledge includes whatever facts/patterns it learned during pretraining, but that knowledge is not guaranteed to be current, point in time, or source traceable.

The LoRA adapter mainly changes **policy**: how the model behaves when a prompt supplies evidence and asks for a bounded financial answer.

For example, the adapter can increase the probability of a sequence that:

- states one conclusion;
- marks materiality;
- separates factual claims from interpretation;
- cites only supplied evidence IDs;
- expresses uncertainty explicitly.

This is a much more stable training target than trying to memorize current market levels.

Imagine two approaches to a new CPI release.

**Memorization approach:** fine-tune the model so it remembers the new CPI number.

**Evidence-use approach:** keep the model fixed and supply the new CPI release as evidence, relying on the adapter to cite and interpret it.

The second architecture scales better because the behavior remains reusable across every future release.

That is why a relatively small adapter can be important even if it changes only a tiny fraction of total model parameters. It changes the conditional behavior we care about at exactly the point where the model turns evidence into prose.

### 5.2 LoRA is also an experimental-control advantage

A compact adapter makes ablation easier.

We can compare:

- pinned base Qwen with no adapter;
- the same base plus the financial LoRA;
- the merged high-precision model;
- the final Q4 GGUF.

Because the base revision stays fixed, differences between the first two can be attributed much more cleanly to the adapter than if we had fully fine-tuned every parameter and changed several training components simultaneously.

The adapter hash also gives a compact experimental identity. If two runs produce different adapter hashes, we know the learned deltas are not byte-identical.

In future work, this architecture could support several specialized adapters on the same base, for example one focused on filing changes and another on cross-asset synthesis. Whether that would improve the system is an empirical question; the important point is that LoRA makes such modular experiments technically feasible.

### 5.3 Parameter efficiency doesn't remove the need for good data

A common misconception is that LoRA somehow makes small-data fine-tuning safe by construction.

Low-rank adapters reduce the number of trainable parameters, which can reduce overfitting capacity relative to full fine-tuning. But a rank-16 adapter across many projections and layers still has enough freedom to memorize poor conventions.

If reviewed examples repeatedly:
- overstate causality;
- confuse spread direction;
- use stale evidence;
- treat cash-flow timing as fraud;
- cite numbers without context;

the adapter can learn those habits efficiently.

So the data-quality pipeline — frozen manifests, chronological splits, source hashes, evidence cutoffs and human review — is not secondary to LoRA. It defines what the low-rank update is encouraged to become.

The optimization objective has no independent concept of “financially sensible.” It only rewards matching the supervised target tokens.

### 5.4 LoRA gradients: how two small matrices steer a frozen large matrix

The forward update is

$$
\Delta W=sBA,\qquad s=\frac{\alpha}{r}.
$$

Let the upstream derivative of the loss with respect to the layer output be $G=\partial\mathcal L/\partial Y$. The layer output is

$$
Y = WX+sBAX.
$$

Because $W$ is frozen, we care about the adapter gradients. Using matrix calculus schematically,

$$
\frac{\partial \mathcal L}{\partial B}
=
s\,G(AX)^\top,
$$

and

$$
\frac{\partial \mathcal L}{\partial A}
=
s\,B^\top GX^\top
$$

with batch/sequence dimensions folded appropriately.

This shows something useful: $A$ learns a low-dimensional projection of the incoming representation, and $B$ learns how those rank-$r$ features should be written into the output space.

If $r=16$, every adapted projection can use at most a rank-16 correction in one linear operation. Across many layers and several target matrices, however, those corrections compound through nonlinearities and residual connections. A small parameter count can therefore create a meaningful change in end-to-end behavior.

### Alpha and rank

The scaling factor is $s=\alpha/r$. Here $\alpha=r=16$, so $s=1$. If we increased rank while holding alpha fixed, each individual update would be scaled down. Many LoRA recipes use this scaling so rank changes don't automatically explode the adapter contribution.

Rank is a capacity choice:

- very small $r$: cheap but may underfit the behavioral shift;
- very large $r$: more capacity and optimizer state, but higher memory/cost and more room to overfit a small supervised dataset.

Rank 16 is a moderate adapter for a 2B model. We shouldn't claim it is globally optimal without a rank sweep; we treat it as a fixed reviewed recipe.

### Why target both attention and MLP projections

Imagine the prompt contains:

> Revenue rose 18%, operating income rose 19%, CFO fell 52%.

An analyst response has to identify the cash-flow tension, connect related numbers, and express uncertainty. Attention adapters can change which prompt positions are emphasized when later answer tokens are produced. MLP adapters can change the nonlinear feature mapping that turns those attended patterns into representations associated with “cash conversion weakness,” “timing,” or “working-capital investigation.”

Restricting LoRA to only Q/V, for example, would be a legitimate alternative recipe, but it would reduce the adaptation degrees of freedom. Our code explicitly targets Q/K/V/O plus gate/up/down, and the run identity hashes that choice.

### Frozen doesn't mean computationally free

The base model still performs almost all forward computation. LoRA reduces:

- trainable parameter memory;
- gradient storage for base weights;
- optimizer state for base weights;
- checkpoint size.

It doesn't reduce the cost of computing 24 Qwen layers on an 8K sequence to zero. That is why a substantial GPU is still required despite parameter-efficient tuning.

#### Low rank constrains the update, not the complexity of the final response

The rank restriction is easiest to understand geometrically. For one adapted projection,

$$
W' = W + \Delta W, \qquad \Delta W = \frac{\alpha}{r}BA,
$$

and therefore

$$
\operatorname{rank}(\Delta W)\le r.
$$

If $r=16$, the optimizer cannot move that layer's full weight matrix in an arbitrary high-dimensional direction. It can only build an update whose column and row spaces are generated by the learned low-rank factors.

That sounds restrictive, but several facts make it much less limiting than it first appears.

First, we are **not learning the language model from zero**. The frozen Qwen base already contains token representations, syntax, broad world knowledge, instruction-following behavior and the ability to produce coherent text. We mainly want to alter how those capabilities are used for a narrow financial-analysis protocol: cite evidence, respect cutoffs, separate facts from interpretations, produce a fixed schema and avoid unsupported numerical claims.

Second, LoRA is inserted into many projections across many layers. A rank-16 change in one matrix is small, but a sequence of low-rank changes separated by nonlinear activations, normalization, residual connections and attention can change the network's overall function in a much richer way. If layer $\ell$ applies

$$
h_{\ell+1}=f_\ell(h_\ell;W_\ell+\Delta W_\ell),
$$

then changing several $\Delta W_\ell$ changes the trajectory of hidden states through the whole stack. The final function isn't itself restricted to a rank-16 transformation of the input.

Third, the useful adaptation directions for a specialized task may occupy a much smaller subspace than the full parameter space. Imagine that the base model already knows the English concepts of revenue, liquidity, inflation and uncertainty. We don't need to relearn those concepts. We may mainly need to strengthen a set of behaviors such as:

- attend to evidence IDs near numerical claims;
- prefer a compact JSON structure;
- state uncertainty when evidence conflicts;
- avoid adding a number that isn't in the packet;
- distinguish a factual observation from an analyst interpretation.

A low-dimensional update can be enough to shift those recurring behaviors.

There is still no theorem saying rank 16 is optimal for our dataset. A higher rank gives more adaptation capacity but adds trainable parameters, optimizer memory and overfitting opportunity. A lower rank is cheaper but can underfit if the behavior change is too complex. Our recipe fixes $r=16$ as a reviewed engineering choice rather than pretending it was discovered by a broad hyperparameter search.

A useful diagnostic if training underperforms would be to ask **what kind of error remains** before increasing rank. If JSON structure is weak, data formatting or target consistency may be the real problem. If all tasks improve except one highly specialized task, task balance may be the problem. If validation loss plateaus early across every task and the adapter clearly lacks capacity, then rank becomes a more plausible bottleneck.

That sequence of diagnosis keeps LoRA rank from becoming a magic tuning knob.

### 5.5 Parameter efficiency, optimizer memory and why this recipe fits one training GPU

Suppose full fine-tuning exposed $P$ parameters. AdamW typically needs, conceptually:

- parameter values;
- gradients;
- first moment $m$;
- second moment $v$;

often with some states kept at higher precision depending on the implementation.

The memory tied to optimization therefore grows by several multiples of the trainable parameter count. For a 2B model, full fine-tuning can become much more memory intensive than merely loading 2B inference weights.

With LoRA, let $P_L\ll P$ be the adapter parameter count. Optimizer state scales with $P_L$:

$$
M_{\text{optimizer}}\propto P_L,
$$

while the frozen base mostly contributes its 16-bit weights and activation computation.

Activation memory can still dominate at long sequence lengths, which is why the recipe combines:

- micro-batch 1;
- gradient accumulation 8;
- gradient checkpointing;
- 8,192-token maximum;
- BF16/FP16 base compute.

These techniques solve different constraints. Gradient accumulation addresses batch-memory pressure. Checkpointing addresses activation-memory pressure. LoRA addresses trainable-weight and optimizer-memory pressure. Mixed precision addresses weight/activation bandwidth and compute efficiency.

Using all four is coherent rather than redundant.

### Why adapters stay in FP32

A rank-16 adapter is tiny relative to the base. Keeping $A$ and $B$ in FP32 costs little memory but gives gradients and updates more mantissa precision.

Think of the final dense correction as

$$
\Delta W = BA.
$$

If the learned changes are small relative to base weight magnitudes, coarse rounding in the adapter could erase part of the signal we are trying to learn. FP32 adapters avoid that unnecessary source of noise during optimization.

At merge time, the correction is combined with 16-bit base weights. The final deployment quantization later introduces its own approximation, which we test separately.

### 5.6 Loading the exact trainable parameter set

The model-loading cell applies Unsloth's adapter machinery, turns on gradient checkpointing and then audits every parameter with `requires_grad=True`.

The only trainable tensors should be LoRA tensors. If a vision layer, base language matrix or visual projector appears in that list, the cell raises an error.

This check converts our intended method into an invariant:

$$
\Theta_{\text{train}}
=
\Theta_{\text{LoRA only}}.
$$

It also checks adapter dtype. Keeping small trainable matrices in FP32 gives the optimizer more numerical precision even while the expensive frozen base computations use BF16/FP16.

If we resume from a saved adapter, the code loads that adapter against the exact base and checks missing/unexpected keys. A structurally mismatched adapter should fail here rather than produce a subtly broken merged model later.

A successful execution reports the trainable-parameter count and GPU memory use. Those are the numbers we would use to quantify parameter efficiency once the cell is actually run.

In [ ]:
def load_model(adapter_path=None):
    log("Loading model", weights="saved adapter" if adapter_path else "pinned base")
    set_seed(seed)
    model, processor = FastVisionModel.from_pretrained(
        model_name=base_model, revision=base_revision, cache_dir=str(cache_dir),
        max_seq_length=training_context, load_in_4bit=False, dtype=dtype,
        use_gradient_checkpointing="unsloth", full_finetuning=False,
    )
    model = FastVisionModel.get_peft_model(
        model, finetune_vision_layers=False, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        target_modules=lora_config["target_modules"], r=lora_config["rank"],
        lora_alpha=lora_config["alpha"], lora_dropout=lora_config["dropout"],
        bias=lora_config["bias"], random_state=seed,
    )
    model.peft_config["default"].revision = base_revision
    model.peft_config["default"].base_model_name_or_path = base_model
    trainable = [(name, param) for name, param in model.named_parameters() if param.requires_grad]
    if not trainable or any("lora_" not in name for name, _ in trainable):
        raise ValueError("Only LoRA parameters should be trainable.")
    for name, param in trainable:
        if any(part in name.lower() for part in ["visual", "vision", "in_proj", "out_proj"]):
            raise ValueError(f"Unexpected trainable projection: {name}")
        if not any(f".{target}." in name for target in lora_config["target_modules"]):
            raise ValueError(f"Unexpected LoRA target: {name}")
        param.data = param.data.float()
    if adapter_path is not None:
        from peft import set_peft_model_state_dict
        from safetensors.torch import load_file
        weights = load_file(str(Path(adapter_path) / "adapter_model.safetensors"))
        loaded = set_peft_model_state_dict(model, weights, adapter_name="default")
        if loaded.unexpected_keys or any("lora_" in key for key in loaded.missing_keys):
            raise ValueError("Saved adapter weights do not match the configured LoRA modules.")
        model.requires_grad_(False)
    if getattr(model, "is_loaded_in_4bit", False):
        raise RuntimeError("This run requires a 16-bit base model.")
    for config in [model.config, model.config.text_config, model.generation_config]:
        config.eos_token_id = tokenizer.eos_token_id
        config.pad_token_id = tokenizer.pad_token_id
    log("Model loaded", lora_parameters=sum(param.numel() for _, param in trainable),
        base_precision=str(dtype), adapter_precision="float32", gpu_gib=round(torch.cuda.memory_allocated() / 2**30, 2))
    return model

model = load_model(adapter_dir if trained else None)
model.config.use_cache = False

## 6. Optimization and the Training Run

The trainer now brings together the objective, optimizer and batching rules.

For one example with supervised token set $\mathcal S$, completion-only cross-entropy is

$$
\mathcal L(\theta)
=
-\frac{1}{|\mathcal S|}
\sum_{t\in\mathcal S}
\log p_\theta(y_t\mid x_{<t}),
$$

where $\theta$ contains only LoRA parameters from the optimizer's point of view.

Backpropagation applies the chain rule through every frozen layer needed to compute the loss, but only derivatives with respect to $A$ and $B$ become parameter updates.

### Gradient accumulation

Let $g_i$ denote the gradient from micro-batch $i$. With eight accumulation steps, the optimizer update is based on the accumulated/appropriately normalized gradient

$$
g_{\text{eff}}
\propto
\sum_{i=1}^{8}g_i.
$$

We get a batch-like estimate without holding eight 8K-token examples in memory simultaneously.

Accumulation does **not** make batch size one identical to true batch eight in every implementation detail. Dropout, normalization choices and optimizer stepping can matter. Here LoRA dropout is zero and transformer normalization is tokenwise rather than batch-statistical, so the approximation is especially natural.

### AdamW on adapter parameters

Projects 19 and 20 already introduced adaptive optimization, so we only need the role it plays here. AdamW keeps moving first and second moments of the adapter gradients:

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,
$$

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
$$

Bias-corrected moments scale the update, while decoupled weight decay shrinks parameters separately. The learning rate is $5\times10^{-5}$.

Only LoRA matrices receive this optimizer state. That is another major memory saving relative to full fine-tuning: Adam-style optimizers often require multiple auxiliary values per trainable parameter.

### Warmup

The scheduler warms the learning rate during the first 10% of steps. Early in fine-tuning, the adapter is near its initialization and gradients can be poorly calibrated. Warmup avoids jumping immediately to the full step size.

If the peak learning rate is $\eta_{\max}$ and warmup lasts $T_w$ steps, a simple linear warmup resembles

$$
\eta_t
=
\eta_{\max}\frac{t}{T_w},
\qquad t\le T_w.
$$

The library then follows its configured schedule afterward.

### Gradient clipping

The maximum gradient norm is 1. If the raw gradient has norm $\|g\|_2>1$, clipping rescales it:

$$
g_{\text{clip}}
=
g\frac{1}{\|g\|_2}.
$$

More generally with threshold $c$,

$$
g_{\text{clip}}=g\min\left(1,\frac{c}{\|g\|_2}\right).
$$

This doesn't solve a systematically bad learning rate, but it limits isolated exploding updates.

### Mixed precision and loss scaling

FP16 can underflow small gradients. Automatic mixed precision may scale the loss before backward, then unscale gradients before the optimizer step. If overflow is detected, the update is skipped and the scale is reduced.

The custom callback counts those skipped updates. One occasional skip can happen. Eight consecutive overflow-related skips cause an abort because optimization is no longer progressing normally.

With BF16, dynamic loss scaling is often unnecessary because the exponent range is much larger, but we keep the monitoring logic appropriate to the selected precision.

### Evaluation and early stopping

Every 50 optimizer steps, the trainer measures validation loss and saves a checkpoint. The best checkpoint is selected by `eval_loss`.

Early stopping patience is two evaluation rounds with threshold $10^{-4}$. The intent is to stop if chronological validation performance no longer improves enough, rather than forcing both epochs merely because they were scheduled.

A lower training loss with rising validation loss would be the classic overfitting pattern. In this application, validation loss is only one gate: the final adapter also has to pass behavior-specific acceptance checks.

### 6.1 What one optimizer update means end to end

It is useful to trace one update without code details.

A reviewed example enters as a serialized chat sequence. The tokenizer maps that sequence into IDs. Qwen converts IDs into embeddings and propagates them through its hybrid language stack. At every supervised answer position, the model produces logits over the vocabulary.

Cross-entropy compares those logits with the next gold answer token. Backpropagation propagates error information through the whole computation graph until it reaches the LoRA matrices attached to attention and MLP projections. The frozen base weights participate in the forward and backward computation but receive no optimizer update.

Because the micro-batch is one, this process repeats for several examples. Their gradients accumulate. After eight micro-batches, gradients are unscaled if mixed precision requires it, checked for finite values, clipped if their global norm exceeds the threshold, and passed to AdamW.

AdamW updates only adapter parameters. The scheduler advances the learning rate. The gradients are cleared and the next accumulation window begins.

After the configured number of update steps, the trainer pauses optimization and measures validation loss on later examples. When a checkpoint is due, it saves enough state to resume the same trajectory.

This sequence clarifies several terms that are easy to mix:

- **forward pass:** compute predictions/loss;
- **backward pass:** compute gradients;
- **micro-batch:** one sequence processed before accumulation;
- **optimizer step:** one actual parameter update after several micro-batches;
- **evaluation step:** inference-like loss measurement with no weight update;
- **checkpoint:** saved training state, not merely exported adapter.

When the log says “step 50,” it refers to optimizer updates, not 50 individual examples.

### 6.2 Why completion-only loss is especially appropriate for RAG-style supervision

The prompt can contain thousands of evidence tokens. If we trained the model to predict those tokens too, most of the loss could be dominated by reproducing source text that the model is supposed to *read*, not memorize.

Completion-only loss allocates gradient to the analytical response.

The prompt still determines hidden states. A wrong evidence passage can therefore still teach the wrong mapping, but the optimization target is the assistant's behavior conditional on that passage.

This mirrors deployment. Project 24 will construct a fresh evidence packet, then ask the model to generate only the answer. Training and application share the same conditional direction:

$$
\text{evidence}\rightarrow\text{analysis}.
$$

That alignment is more important than maximizing the amount of supervised token text.

### 6.3 Sequence length, token weighting and what the objective emphasizes

Cross-entropy is accumulated over supervised answer tokens. That means the effective weighting of examples depends on how the trainer aggregates token losses.

If example $i$ has $A_i$ supervised tokens, a token-averaged objective resembles

$$
\mathcal L
=
-\frac{1}{\sum_i A_i}
\sum_i\sum_{t=1}^{A_i}
\log p_\theta(y_{i,t}\mid x_i,y_{i,<t}).
$$

Long completions can therefore contribute more individual token predictions than short completions.

That is not automatically bad. A longer answer contains more behavior to learn: schema fields, claims, citations and uncertainty. But it explains why “number of examples” is not the same as “amount of supervision.”

Our frozen examples use a consistent structured schema, which reduces extreme variation. Still, when we inspect task balance, we should remember there are at least three relevant distributions:

- number of examples per task;
- prompt tokens per task;
- supervised completion tokens per task.

A task with fewer but much longer completions can carry substantial gradient weight.

### Exposure bias and why repair logic remains useful

Teacher forcing optimizes

$$
p(y_t\mid x,y_{<t}^{\text{gold}}).
$$

At inference we instead condition on the model's own generated prefix:

$$
p(\hat y_t\mid x,\hat y_{<t}).
$$

If an early token is wrong, later predictions see a prefix that may never have appeared in supervised data. This is usually called exposure bias.

The answer schema helps. JSON gives a strong regular structure, and constrained decoding narrows legal next tokens. Repair gives one more chance when semantic validators fail.

We don't need to solve exposure bias with a new algorithm here. The system architecture can compensate by combining several simple controls around the language model.

### Why two epochs can be enough

A small high-quality behavior dataset doesn't necessarily benefit from many epochs. The base model already has strong next-token competence. Fine-tuning is trying to rotate behavior toward a narrow conditional distribution.

Too many passes can increase memorization of exact sentence forms or examples. Chronological validation and early stopping therefore matter more than maximizing the number of epochs.

If the adapter fails acceptance after two epochs, the first question shouldn't automatically be “train longer.” We would inspect which failures occur:

- schema failures may need better format examples or decoding constraints;
- unsupported numbers may need stronger traceability examples;
- bad financial interpretations may need better supervision/data, not more repetitions of the same data;
- underfitting across all tasks may justify capacity or optimization changes.

### 6.4 Learning rate, curvature and why LoRA can still diverge

Parameter-efficient tuning is smaller than full fine-tuning, but its loss surface is still nonlinear. The adapter parameters enter many layers, and their effects compound through residual blocks.

A first-order update is

$$
\theta_{t+1}=\theta_t-\eta_t\hat g_t.
$$

If $\eta_t$ is too large relative to local curvature, a low-rank adapter can still push the model into a region with unstable logits or catastrophic behavior on the task.

Gradient clipping limits individual steps, warmup reduces early shocks, and finite-loss/AMP checks catch numerical failure. They don't replace validation.

The chosen $5\times10^{-5}$ rate is therefore best understood as part of this frozen recipe, not a universal LoRA constant.

### Weight decay in a low-rank adapter

AdamW applies decoupled shrinkage roughly as

$$
\theta_{t+1}
=
(1-\eta_t\lambda)\theta_t
-
\eta_t\cdot\text{AdamStep}(g_t),
$$

with $\lambda=0.01$ here.

Because the base is frozen, weight decay acts on adapter tensors, encouraging the learned delta not to grow unnecessarily large. The base model itself isn't shrunk.

### Validation loss and task utility can disagree

Suppose an adapter gets much better at predictable JSON punctuation and common wording while financial interpretation changes little. Token loss can improve.

Conversely, one rare but important materiality decision might improve while having almost no visible effect on aggregate loss.

That is why the evaluation hierarchy is:

1. numerical training health;
2. chronological validation loss;
3. structured acceptance checks;
4. human financial review.

Each layer answers a different question.

### 6.5 The finite-loss and overflow guards

The custom trainer checks every computed loss with

$$
\operatorname{isfinite}(\mathcal L).
$$

`NaN` or `inf` stops training immediately.

That may look severe, but continuing after a non-finite objective can contaminate optimizer moments and adapter weights. A saved checkpoint after numerical failure could look like an ordinary file while containing unusable parameters.

The training callback also records peak GPU memory and skipped AMP updates. These diagnostics answer a different question from loss: **did the optimization execute numerically as intended?**

Once this cell has run, a `Trainer` exists with the complete policy for optimization, evaluation, checkpointing and early stopping. No update occurs until the next cell calls `train()`.

In [ ]:
class FiniteLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        result = super().compute_loss(model, inputs, return_outputs=return_outputs,
                                      num_items_in_batch=num_items_in_batch)
        loss = result[0] if return_outputs else result
        if not torch.isfinite(loss.detach()).all().item():
            raise FloatingPointError("Non-finite loss; restart from the last saved optimizer checkpoint after inspecting the batch.")
        return result

class TrainingProgress(TrainerCallback):
    def __init__(self):
        self.accelerator = None
        self.started = time.monotonic()
        self.skipped_updates = 0
        self.consecutive_skips = 0

    def on_step_end(self, args, state, control, **kwargs):
        skipped = self.accelerator.optimizer_step_was_skipped
        self.skipped_updates += int(skipped)
        self.consecutive_skips = self.consecutive_skips + 1 if skipped else 0
        if skipped:
            log("AMP skipped an overflowed update and reduced its scale", step=state.global_step,
                scale=self.accelerator.scaler.get_scale())
        if self.consecutive_skips >= 8:
            raise FloatingPointError("Eight consecutive AMP skips; inspect numerical stability before resuming.")

    def on_log(self, args, state, control, logs=None, **kwargs):
        row = {"step": state.global_step, "elapsed_seconds": round(time.monotonic() - self.started, 1),
               "amp_skips_this_session": self.skipped_updates,
               "gpu_gib": round(torch.cuda.memory_allocated() / 2**30, 2), **(logs or {})}
        with (run_dir / "training_log.jsonl").open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(row) + "\n")
        log("Training progress", **row)

    def on_save(self, args, state, control, **kwargs):
        log("Model, optimizer, scheduler, RNG and scaler checkpoint saved", step=state.global_step)

progress = TrainingProgress()
arguments = TrainingArguments(output_dir=str(checkpoints_dir), num_train_epochs=epochs,
    per_device_train_batch_size=batch_size, per_device_eval_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation, learning_rate=learning_rate,
    warmup_ratio=0.1, weight_decay=0.01, max_grad_norm=1.0, optim="adamw_torch",
    fp16=not bf16, bf16=bf16, logging_steps=5, logging_first_step=True, logging_nan_inf_filter=False,
    eval_strategy="steps", eval_steps=evaluation_steps, prediction_loss_only=True,
    save_strategy="steps", save_steps=evaluation_steps, save_total_limit=3, save_only_model=False,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    restore_callback_states_from_checkpoint=True,
    seed=seed, data_seed=seed, dataloader_num_workers=0, report_to="none", remove_unused_columns=False)
callbacks = [progress]
if early_stopping_patience:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience,
                                          early_stopping_threshold=0.0001))
trainer = None
if trained is None:
    FastVisionModel.for_training(model)
    trainer = FiniteLossTrainer(model=model, args=arguments, train_dataset=encoded["train"],
        eval_dataset=encoded["validation"], data_collator=collator, processing_class=tokenizer, callbacks=callbacks)
    progress.accelerator = trainer.accelerator
    if not bf16 and last_checkpoint is None:
        trainer.accelerator.scaler = torch.amp.GradScaler("cuda", init_scale=1024)

### 6.6 Executing the Single Adaptation Phase

This cell performs the only model-training phase in this workflow:

```text
trainer.train(...)
```

If a compatible checkpoint exists, it resumes. Otherwise it starts from the pinned base plus initialized LoRA parameters.

Each optimizer update conceptually follows:

1. tokenize/pad the next micro-batch;
2. run the causal forward pass;
3. compute completion-only cross-entropy;
4. backpropagate into LoRA parameters;
5. accumulate gradients until eight micro-batches are collected;
6. unscale/check/clip gradients as required;
7. apply AdamW;
8. update the learning-rate schedule;
9. periodically evaluate and checkpoint.

Two epochs mean the examples may be seen twice unless early stopping ends the process sooner.

### What a falling loss actually means

If the training loss falls, the adapted model is assigning more probability to the reviewed completions given their prompts:

$$
\mathcal L\downarrow
\quad\Rightarrow\quad
\prod_{t\in\mathcal S}p(y_t\mid x_{<t})\uparrow
$$

in aggregate.

That is useful but incomplete. It doesn't directly prove that:

- citations refer to the correct evidence;
- an interpretation is financially sound;
- unsupported numbers never appear;
- JSON is always valid during free generation;
- a quantized model will preserve the behavior.

Those are tested later.

### Saving the adapter rather than the full base

After a successful run, we save the LoRA adapter in `safetensors` format along with the tokenizer, trainer state and a `trained.json` receipt. The receipt includes the adapter hash, scheduled updates, training loss, best checkpoint/eval loss and AMP-skip information.

The compact adapter is the **learned delta**. The 2B pretrained base can remain a separately pinned dependency until merge time.

Because the attached training notebook contains no stored outputs, we shouldn't state a final loss or number of updates here. When the cell is actually run, those values are the first quantitative evidence we can analyze.

### 6.7 Early stopping is model selection, not an optimization failure

Suppose evaluation losses at successive checkpoints are

$$
1.10,\quad 0.94,\quad 0.88,\quad 0.881,\quad 0.884.
$$

Training loss might still be falling. The validation sequence says the later updates are no longer helping the chronological holdout.

With patience two, the trainer allows short plateaus/noise rather than stopping after the first non-improvement. The threshold $10^{-4}$ prevents tiny floating-point changes from being treated as meaningful gains.

`load_best_model_at_end=True` means the adapter we export should correspond to the best recorded evaluation checkpoint, not necessarily the last gradient step.

This is useful for LoRA because the adapter can over-specialize even though only a small fraction of parameters move. Parameter efficiency reduces memory; it doesn't eliminate overfitting.

### Why we don't tune on the acceptance examples

The thirty acceptance cases are kept for the release gate. If we repeatedly changed hyperparameters until those exact cases all passed, they would become a de facto tuning set.

In a larger research pipeline we might maintain separate train, validation, acceptance and final test sets. Here we at least separate optimization by validation loss from downstream structural/financial acceptance checks and records exactly which examples form that gate.

In [ ]:
if trained is None:
    log("Starting the single training phase", resume=last_checkpoint, early_stopping_patience=early_stopping_patience)
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
    if not np.isfinite(result.training_loss):
        raise FloatingPointError("Final training loss is not finite.")
    adapter_dir.mkdir(exist_ok=True)
    model.save_pretrained(str(adapter_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(adapter_dir))
    trainer.state.save_to_json(str(run_dir / "trainer_state.json"))
    trained = {"adapter_sha256": sha256(adapter_dir / "adapter_model.safetensors"),
               "scheduled_updates": trainer.state.global_step, "training_loss": result.training_loss,
               "best_checkpoint": trainer.state.best_model_checkpoint, "best_eval_loss": trainer.state.best_metric,
               "amp_skips_this_session": progress.skipped_updates, "step_count_includes_amp_skips": True}
    save_json(trained_path, trained)
    log("Completed adapter saved", path=adapter_dir)
else:
    log("Completed adapter reused; training skipped")
display(pd.Series(trained))

### 6.8 What we would inspect after training finishes

The run should end in one of three meaningful states:

**Normal completion.** Training reaches the scheduled end, losses remain finite, the best checkpoint is recorded and the adapter is hashed.

**Early stop.** Validation loss stops improving enough and the trainer restores the best checkpoint. This can be preferable to completing both epochs.

**Hard failure.** Non-finite loss, repeated AMP overflow, identity mismatch or another guard aborts the run. We shouldn't export an adapter from this state.

The saved `trained.json` becomes the bridge between optimization and acceptance testing. Downstream export cells require a verified adapter identity instead of merely checking that a directory exists.

### 6.9 Reading Training and Validation Loss

The next cell loads `trainer_state.json` and plots logged training loss and evaluation loss against optimizer step.

Training loss is measured on examples the optimizer is actively fitting. Validation loss comes from later chronological examples whose tokens don't update the model.

A healthy small-data fine-tune often shows:

- training loss decreasing;
- validation loss decreasing initially;
- eventually a plateau or mild reversal in validation while training can keep improving.

If validation loss starts rising strongly while training loss continues down, the adapter is specializing too tightly to the training completions.

We should also read loss scale carefully. Cross-entropy averages over supervised tokens. A few long answers can contribute many token predictions, and common JSON syntax can be easy while rare financial reasoning tokens are hard. One scalar loss doesn't tell us which behavior improved.

Perplexity is sometimes reported as

$$
\operatorname{PPL}=e^{\mathcal L}.
$$

For a completion-only, task-specific dataset, perplexity is not the main business metric. A lower value says the gold tokens became more probable. It doesn't tell us whether a generated financial conclusion is useful.

### Switching from training to inference

After plotting, we free trainer/optimizer state, runs garbage collection, releases CUDA cache, switches the model into inference mode and enables caching.

The memory profile changes substantially:

- optimizer moments disappear;
- gradients are no longer stored;
- dropout-like training behavior is disabled;
- K/V or recurrent sequence caches can be used for generation.

That lets the same GPU run deterministic acceptance generation without carrying training-state memory.

### 6.10 Diagnosing the shape of the loss curves

When we finally have a run, several patterns have different meanings.

**Training and validation fall together.**  
The adapter is learning target behavior that transfers to later examples. We still need acceptance tests, but this is the cleanest pattern.

**Training falls while validation stays flat.**  
The adapter may be fitting idiosyncratic wording in the training set. If acceptance tasks are also flat or weak, more epochs are unlikely to help.

**Validation improves early and then rises.**  
That supports using the best checkpoint rather than final weights. Early stopping exists precisely for this case.

**Both losses spike or become noisy.**  
We would examine learning rate, AMP overflows, pathological long examples, and data errors before interpreting the run economically.

**Loss is almost constant from the start.**  
Possible explanations include a very small adapter gradient, a masking bug, a frozen adapter, a learning rate that is too low, or a task already nearly solved by the base. The collator/trainable-parameter unit tests help eliminate the first three.

### Comparing checkpoints fairly

Because validation is chronological, checkpoint comparison holds the validation examples fixed while only adapter weights change. We should not select a checkpoint using the later Project 24 live questions and then claim those same questions are out-of-sample evaluation. They are application examples, not training-set model selection.

### Loss scale and confidence

Cross-entropy has no natural “good financial analyst” threshold. A loss of 0.8 isn't inherently a passing value and 1.1 isn't inherently failing. Token entropy depends on vocabulary, answer style and dataset consistency.

We therefore avoid arbitrary loss cutoffs. We use **relative validation improvement** to select checkpoints and concrete behavioral validators to decide export.

In [ ]:
state = json.loads((run_dir / "trainer_state.json").read_text(encoding="utf-8"))
history = pd.DataFrame(state["log_history"])
fig, ax = plt.subplots(figsize=(9, 3.5))
for column in ["loss", "eval_loss"]:
    if column in history:
        history.dropna(subset=[column]).plot(x="step", y=column, marker=".", ax=ax)
ax.set(title="Completion-only training and validation loss", ylabel="Loss", xlabel="Scheduled update")
plt.show()
if trainer is not None:
    trainer.optimizer = None
    trainer.lr_scheduler = None
del trainer
progress.accelerator = None
gc.collect()
torch.cuda.empty_cache()
FastVisionModel.for_inference(model)
model.config.use_cache = True

### 6.11 How we would interpret the actual curve

Because the source notebook hasn't been executed with outputs saved, the correct markdown here is a reading guide rather than a fabricated diagnosis.

When a real run is available, we should record:

- best validation-loss step;
- whether early stopping fired;
- distance between final training and best validation loss;
- any sharp loss spikes;
- whether validation evaluations are too sparse to locate a stable optimum.

If only one or two evaluation points exist because the dataset is tiny, early-stopping conclusions are weak. The later acceptance suite then becomes even more important.

## 7. From a Trained Adapter to Accepted Financial Responses

Training defines a probability distribution. **Decoding** decides how we turn that distribution into an answer.

The acceptance generator is deterministic:

- `do_sample=False`;
- repetition penalty 1.0;
- fixed max-new-token budget;
- fixed maximum generation time;
- thinking disabled.

With greedy decoding, each step chooses

$$
\hat x_{t+1}=\arg\max_j p_\theta(j\mid x_{\le t}).
$$

There is no temperature sampling. Re-running the same model/prompt under the same deterministic runtime should therefore be much more reproducible.

The function tokenizes the prompt first and verifies that enough context remains for the answer. It requires at least 512 answer tokens of headroom rather than squeezing a response into an almost-full context.

### EOS and termination

The model should eventually produce EOS. If it hits the token or time limit first, that becomes an explicit validation error.

Termination quality matters in structured analysis. A model that produces valid JSON only because the runtime chopped the output at a convenient brace is not reliable.

### Parsing fenced or raw JSON

Generated text may contain a Markdown code fence around JSON. The parser extracts the JSON payload and validates it against the `Analysis` model.

If the first attempt fails, we construct one repair prompt. The repair receives the original evidence and explicit errors, such as:

- unsupported number;
- invalid evidence ID;
- repeated claim;
- malformed schema.

The model is asked to fix those exact failures rather than regenerate blindly.

We limit repair attempts deliberately. An answer that needs many self-corrections can consume time and eventually stumble into a passing form without demonstrating stable behavior.

### 7.1 Logits, greedy decoding and why deterministic isn't the same as correct

At generation step $t$, the model produces logits $z_t$. Greedy decoding chooses the largest component after any constraints:

$$
\hat y_t=\arg\max_j z_{t,j}.
$$

Because softmax preserves rank, taking argmax of probabilities or logits gives the same token.

Deterministic decoding removes sampling variance, which is useful for a production analyst and for acceptance tests. If an answer fails, we can reproduce the exact failure.

But determinism has a cost: it always follows the model's current highest-probability path. If two interpretations are close and the slightly higher one is wrong, temperature zero won't explore the alternative.

For our system, reproducibility is more valuable than creative diversity. We use explicit evidence, structured tasks and a repair pass instead of stochastic sampling.

### Repetition penalty is neutral here

The setting is `repetition_penalty=1.0`, which means no extra heuristic penalty is applied. We rely on training, schema structure and duplicate validators.

A stronger repetition penalty can reduce loops but can also distort required repeated tokens in JSON keys, source IDs or financial terms. Keeping it neutral makes the acceptance behavior easier to attribute to the model and constraints.

### Time and token budgets

Generation can stop for several reasons:

- EOS;
- maximum new tokens;
- maximum wall-clock time;
- external stopping criterion.

Only EOS is the desired semantic completion. The validator records budget exhaustion as an error.

That distinction becomes important on local hardware. A slow model that reaches `max_time` may produce syntactically incomplete JSON even if its reasoning was otherwise fine. We shouldn't label that a financial failure.

### One repair attempt as error-conditioned generation

The repair prompt adds information about the first failure:

$$
p_\theta(
y_{\text{repair}}
\mid
x,\hat y_{\text{first}},e_{\text{validation}}
).
$$

The model gets the original evidence plus explicit errors. This is different from asking the same question twice.

If an unsupported number appears, the repair can remove or replace it. If an evidence ID is invalid, it can select from the packet. Limiting repairs to one keeps the system bounded and exposes unstable cases instead of hiding them behind repeated retries.

In [ ]:
generation_settings = {"do_sample": False, "repetition_penalty": 1.0, "max_new_tokens": generation_tokens,
                       "max_time": generation_seconds, "check_version": "financial-checks-1"}

class GenerationProgress(StoppingCriteria):
    def __init__(self, prompt_length, label, budget):
        self.prompt_length = prompt_length
        self.label = label
        self.budget = budget
        self.started = time.monotonic()
        self.continue_generation = torch.zeros(1, dtype=torch.bool, device="cuda")

    def __call__(self, input_ids, scores, **kwargs):
        count = input_ids.shape[1] - self.prompt_length
        if count % 64 == 0:
            elapsed = time.monotonic() - self.started
            print(f"  {self.label}: {count}/{self.budget} tokens, {elapsed:.0f}s, {count / max(elapsed, 0.001):.1f} tokens/s", flush=True)
        return self.continue_generation

def decode_result(raw, row):
    output = raw.strip()
    fence = re.fullmatch(r"```(?:json)?\s*(\{.*\})\s*```", output, flags=re.S)
    if fence:
        output = fence.group(1).strip()
    return {"output": output, "raw_output": raw, **check_output(output, row)}

def generate(model, row, messages=None):
    messages = row["messages"][:2] if messages is None else messages
    inputs = tokenizer(prompt_text(messages), return_tensors="pt", add_special_tokens=False).to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    budget = min(generation_tokens, training_context - prompt_length)
    if budget < 512:
        raise ValueError("The prompt leaves insufficient room for a complete answer.")
    meter = GenerationProgress(prompt_length, row["example_id"], budget)
    started = time.monotonic()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=budget, max_time=generation_seconds,
            do_sample=False, repetition_penalty=1.0, use_cache=True,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=StoppingCriteriaList([meter]), return_dict_in_generate=False)
    answer = output[0, prompt_length:]
    stopped_eos = bool(len(answer) and answer[-1].item() == tokenizer.eos_token_id)
    elapsed = time.monotonic() - started
    result = {**decode_result(tokenizer.decode(answer, skip_special_tokens=True), row),
              "generated_tokens": len(answer), "stopped_eos": stopped_eos, "seconds": round(elapsed, 2)}
    if not stopped_eos:
        result["errors"].append("Generation reached its token or time budget before EOS")
    return result

def repair_messages(row, first):
    correction = ("Correct the answer using only the original evidence. Errors: " + json.dumps(first["errors"])
                  + ". Return a complete, concise JSON object. Remove unsupported numbers and repeated claims.")
    return [*row["messages"][:2], {"role": "assistant", "content": first["output"]},
            {"role": "user", "content": correction}]

### 7.2 Expected behavior of the acceptance generator

For each validation example, we will later see:

- whether the first answer parsed;
- whether EOS was reached;
- whether duplicate/traceability/reference checks passed;
- whether repair was needed;
- how long generation took.

The code doesn't use these checks to train again. They are **acceptance tests** after optimization. That separation keeps evaluation from quietly becoming another fine-tuning loop.

### 7.3 Adapter Acceptance Testing

We now evaluate thirty held-out cases: six per task family.

The cache identity includes:

- adapter SHA-256;
- validation-file SHA-256;
- generation settings;
- exact example IDs.

If any of those changes, cached acceptance results are invalid.

For each example:

1. generate once;
2. run schema, reference, duplicate and numerical checks;
3. if necessary, make one constrained repair;
4. store attempts and errors.

The adapter passes only if **all selected examples have no remaining errors**.

This is intentionally stricter than reporting “96% pass rate.” The exported model is going to sit inside a system that expects machine-readable evidence-grounded output. A single known structural failure in this frozen acceptance set should be investigated before deployment.

### What these tests can prove

A pass supports claims such as:

- the output fits the expected JSON schema;
- cited evidence IDs come from the packet;
- fact-like numbers are traceable under the validator;
- responses terminate;
- duplicate behavior is controlled.

### What these tests cannot prove

They cannot prove:

- causality;
- economic interpretation;
- valuation quality;
- materiality judgment;
- completeness;
- sensible prioritization of conflicting signals.

A model can cite a correct number and draw a poor conclusion. Project 24 will show real examples of exactly that boundary.

The cell displays task, schema status, EOS, duplicates and errors for each acceptance example. If any case still fails after its one repair, export stops.

### 7.4 Why task-level acceptance is closer to software testing than to a leaderboard

A generic LLM benchmark might report one score averaged across thousands of questions. Here the model is a component inside a deterministic application, so acceptance looks more like integration testing.

For each case, we know:

- exact prompt;
- exact evidence packet;
- exact allowed source IDs;
- expected structural contract;
- validation rules.

The relevant question is not “is this model state of the art?” It is:

> Can this exact artifact reliably participate in our analyst workflow without violating the contract we designed?

That changes how we treat failures. One malformed JSON response is important even if average token loss is good. One invented evidence ID is important even if 29 other examples pass.

At the same time, we avoid pretending that thirty cases estimate every possible failure probability. They are a frozen release gate, not a complete statistical certification.

### Useful future diagnostics if the gate fails

If failures cluster by task, we can investigate selectively:

- `event`: source/date confusion, causal overreach;
- `sec_change`: comparing wrong periods, missing material changes;
- `macro`: release-level versus trend interpretation;
- `reconciliation`: inability to resolve conflicting evidence;
- `market`: cross-asset synthesis and horizon mismatch.

That diagnosis can guide new reviewed examples. We shouldn't indiscriminately add more of every task if only one behavior is weak.

### Distribution shift remains possible after a clean gate

Project 24 can encounter longer documents, different companies, unusual market regimes or source formats. Validation examples are later in time than training, which helps, but future distribution shift never disappears.

The application responds with layered controls: dated evidence, deterministic feature code, schema checks, caching, response audits and human judgment. Fine-tuning is one layer in that defense.

### 7.5 Why the acceptance gate sits before merging and quantization

Merging and GGUF export are expensive packaging steps. We want to know that the **adapter itself** satisfies the contract first.

If the adapter fails here, quantization isn't the main problem. We should inspect data, training or decoding. If the adapter passes here but the quantized GGUF later fails, the deployment transformation becomes a plausible source of regression.

The two-stage gate therefore helps localize failures.

In [ ]:
checks_path = run_dir / "adapter_checks.json"
check_identity = {"adapter_sha256": trained["adapter_sha256"], "validation_sha256": manifest["files"]["validation.jsonl"],
                  "settings": generation_settings, "examples": [row["example_id"] for row in acceptance_rows]}
saved = json.loads(checks_path.read_text(encoding="utf-8")) if checks_path.exists() else {}
acceptance = saved.get("results", []) if saved.get("identity") == check_identity else []
done = {item["example_id"] for item in acceptance}
for row in acceptance_rows:
    if row["example_id"] in done:
        continue
    log("Checking adapter", case=len(acceptance) + 1, total=len(acceptance_rows), task=row["task"])
    first = generate(model, row)
    attempts = [first]
    if first["errors"]:
        attempts.append(generate(model, row, repair_messages(row, first)))
    acceptance.append({"example_id": row["example_id"], "task": row["task"],
                       "attempts": attempts, **attempts[-1]})
    save_json(checks_path, {"identity": check_identity, "results": acceptance})
display(pd.DataFrame(acceptance)[["task", "schema", "stopped_eos", "duplicate_claims", "errors"]])
adapter_passed = all(not item["errors"] for item in acceptance)
log("Adapter checks complete", passed=adapter_passed, cases=len(acceptance))
if not adapter_passed:
    raise ValueError("Adapter checks failed. Saved weights and outputs remain available; inspect adapter_checks.json before export.")

### 7.6 Human Financial Review

For one held-out item in each task, we print the question, conclusion, claims and uncertainty in a readable form.

This cell is deliberately manual. Automated validators can tell us whether a model cited source `E3`; a human has to decide whether `E3` actually supports the interpretation in context.

A useful review checklist is:

**Evidence selection**
- Did the answer use the strongest available evidence?
- Did it ignore an obvious contradictory source?

**Financial meaning**
- Is a cash-flow change interpreted correctly?
- Is a yield move distinguished from a price move?
- Is correlation mistaken for causality?
- Are levels and changes kept separate?

**Materiality**
- Does “high” reflect economic significance or merely a large percentage from a tiny base?

**Uncertainty**
- Is uncertainty specific to the evidence gap, or generic boilerplate?

**Completeness**
- Did the response answer every part of the question?

We don't turn this into another numeric score here. The point is to keep a place where financial expertise can veto an output that is mechanically valid.

Later, Project 24 applies the same discipline to real model responses. Several pass the automatic checks while still leaving clear analytical weaknesses.

#### Human review also checks omissions

An answer can contain no false claim and still be poor because it leaves out the most important evidence.

Suppose a company packet contains:
- operating margin above 60%;
- free-cash-flow margin above 40%;
- CFO/net-income below 1;
- a large receivables build.

A response that mentions only the receivables build may pass citation and numerical checks, yet give a distorted one-sided view.

So review should ask both:

1. **Is every included claim supported?**
2. **Did the answer include the evidence needed for a balanced conclusion?**

The second question is much harder to automate because it requires judging materiality across the whole packet. Project 24 later exposes exactly this problem in the NVIDIA cash response: the warning is real, but the model underuses strong countervailing profitability and free-cash-flow evidence.

### 7.7 Financial correctness has several levels

During review, it helps to separate four levels that can otherwise blur together.

**Numerical correctness:** the answer copied or used the stated number correctly.

**Accounting/market correctness:** the number is interpreted with the right unit and definition. A 50 bp yield move isn't a 50% price move; CFO isn't free cash flow.

**Economic interpretation:** the relationship makes sense. Rising front-end yields can reflect a more hawkish expected policy path, but we still need the event and curve context before assigning causality.

**Decision relevance:** the answer identifies what deserves attention. A technically true minor detail shouldn't displace a major liquidity or earnings-quality issue.

The automated tests cover much of the first level and pieces of the second. Human review carries most of the third and fourth.

This hierarchy gives us a disciplined way to criticize the model later. We can say “traceable but economically weak” instead of treating an answer as simply correct or wrong.

In [ ]:
for task in tasks:
    item = next(item for item in acceptance if item["task"] == task)
    row = next(row for row in acceptance_rows if row["example_id"] == item["example_id"])
    answer = Analysis.model_validate_json(item["output"])
    display(Markdown("**" + task.replace("_", " ").title() + "**"))
    print(json.loads(row["messages"][1]["content"])["question"])
    print(" ".join(dict.fromkeys([answer.conclusion, answer.what_changed, answer.why_it_matters])))
    display(pd.DataFrame([claim.model_dump() for claim in answer.claims]))
    print("Uncertainty:", " ".join(answer.uncertainty))
print("Automatic checks do not establish financial meaning or causal correctness. Review these answers against their source packets.")

### 7.8 What the review display is expected to give us

The cell should show five readable examples, one per task. We are looking for qualitative consistency across task types, especially because a single aggregate validation loss can hide task-specific weakness.

If a model produces beautiful event summaries but repeatedly mishandles market cross-asset logic, the correct response is not “the adapter passed 30 checks.” We would return to the relevant training examples or system constraints.

## 8. Merging, GGUF Conversion and Local Deployment

A LoRA model at inference can be represented as

$$
y =
\left(
W+\frac{\alpha}{r}BA
\right)x.
$$

We can keep $W$, $A$ and $B$ as separate tensors and add the adapter contribution on every forward pass. Or, after training is finished, we can create a merged matrix

$$
W_{\text{merged}}
=
W+\frac{\alpha}{r}BA.
$$

The next cell performs that merge into 16-bit model weights.

### Why merge before GGUF export

Our deployment target is a single local GGUF file for `llama.cpp`. Merging gives the converter one ordinary model state instead of a base-plus-adapter composition.

After the merge:

- the learned behavior is baked into the dense weights;
- LoRA modules are no longer needed for inference;
- the model can be converted and quantized like an ordinary checkpoint.

Merging is a **linear algebra transformation**, not a second training stage. No loss is computed and no optimizer runs.

The code verifies an identity receipt using the adapter hash, base revision and chat-template hash. If those are unchanged and the merged artifact already exists, it can be reused safely.

The saved form is `merged_16bit`, which is important. We still haven't quantized. Keeping a higher-precision merged checkpoint gives us a clean source for conversion and lets us separate merge errors from quantization errors.

### 8.1 Merging changes storage, not the learned function

Before merge, an adapted linear layer computes

$$
f(x)=Wx+sB(Ax).
$$

After merge,

$$
W_m=W+sBA,
$$

so

$$
f(x)=W_mx.
$$

Ignoring finite-precision rounding, these are algebraically identical.

That is useful operationally. We can train and version a compact adapter, then deploy a standard dense checkpoint without requiring the inference runtime to know LoRA at all.

### Why keep the adapter after merging

The adapter remains valuable even though the deployment file doesn't need it.

If we later change the quantization recipe, we can merge the same verified adapter again. If we upgrade `llama.cpp`, we can reconvert from the same merged identity. If we want to compare the base against the adapted model, the adapter gives us a compact versioned delta.

Deleting it after export would throw away the most interpretable record of what training actually changed.

### Merge precision and numerical equivalence

The adapter matrices are FP32, but the merged checkpoint is saved in 16-bit form. The sum $W+sBA$ is therefore rounded when stored.

That is a first small numerical transformation before Q4 quantization. We don't run a dedicated full acceptance suite at the merged-HF stage in this workflow; the adapter is validated before merge and the final GGUF after all deployment transformations. If a difficult regression appeared later, a diagnostic comparison against the merged or F16 GGUF stage would help isolate it.

In [ ]:
merged_dir = run_dir / "merged"
merge_receipt_path = run_dir / "merge_receipt.json"
merge_identity = {"adapter_sha256": trained["adapter_sha256"], "base_revision": base_revision,
                  "chat_template_sha256": chat_template_hash}
saved_merge = json.loads(merge_receipt_path.read_text(encoding="utf-8")) if merge_receipt_path.exists() else {}
reuse_merge = saved_merge.get("identity") == merge_identity and all(
    (merged_dir / name).is_file() and sha256(merged_dir / name) == digest for name, digest in saved_merge.get("files", {}).items())
reuse_merge = reuse_merge and bool(saved_merge.get("files"))
if not reuse_merge:
    if merged_dir.exists() and any(merged_dir.iterdir()):
        raise ValueError("An incomplete or changed merge exists. Rename that folder before rebuilding it.")
    log("Merging LoRA into 16-bit weights", destination=merged_dir)
    model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
    weights = list(merged_dir.glob("model*.safetensors"))
    if not weights or not (merged_dir / "config.json").is_file() or not (merged_dir / "tokenizer_config.json").is_file():
        raise ValueError("Merged weights or tokenizer files are incomplete.")
    files = {path.name: sha256(path) for path in merged_dir.iterdir() if path.is_file()}
    save_json(merge_receipt_path, {"identity": merge_identity, "files": files})
else:
    log("Verified merged weights reused")
del model
gc.collect()
torch.cuda.empty_cache()
log("Training model released; remaining export runs on CPU")

### 8.2 Expected result of the merge

A successful merge should produce a directory of ordinary merged model weights plus a receipt tying them to the exact adapter and base.

At this point the GPU-heavy training model can be freed. Conversion is intentionally moved to a CPU-oriented, isolated environment later.

If the adapter hash changes, the old merged directory must not be trusted even if filenames are identical.

### 8.3 Isolating the Export Toolchain

Model conversion is a software-build process of its own. We create a dedicated Python 3.12 environment for export tools, installs pinned conversion requirements, clones `llama.cpp`, and checks out the exact pinned commit.

This avoids coupling the training environment to whatever Python packages the current `llama.cpp` converter happens to require.

A useful way to think about the pipeline is:

$$
\text{training environment}
\rightarrow
\text{merged HF checkpoint}
\rightarrow
\text{isolated conversion environment}
\rightarrow
\text{GGUF}
\rightarrow
\text{pinned runtime}.
$$

Each arrow has its own receipt.

The tokenizer is probed in the export environment too. A converter that reads weights correctly but misreads tokenizer metadata could produce a file that loads yet tokenizes prompts differently from training.

In [ ]:
tools_dir = model_dir / "local/export-tools"
tools_dir.mkdir(parents=True, exist_ok=True)
converter_dir = tools_dir / "python312"
converter_python = converter_dir / "bin/python"
converter_probe = "import sys, pip; assert sys.version_info[:2] == (3, 12)"
converter_ready = converter_python.exists() and subprocess.run(
    [str(converter_python), "-c", converter_probe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not converter_ready:
    run_visible([sys.executable, "-m", "pip", "install", "uv==0.8.22"])
    run_visible([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--seed", "--clear", str(converter_dir)])
run_visible([converter_python, "-m", "pip", "install", "-r", model_dir / "requirements-export.in"])
run_visible([converter_python, "-m", "pip", "check"])
run_visible([converter_python, "-c", "import sys; from transformers import AutoTokenizer; t=AutoTokenizer.from_pretrained(sys.argv[1], local_files_only=True); print(type(t).__name__, len(t))", merged_dir])
llama_dir = tools_dir / "llama.cpp"
if not llama_dir.exists():
    run_visible(["git", "clone", "https://github.com/ggml-org/llama.cpp", llama_dir])
run_visible(["git", "-C", llama_dir, "checkout", llama_revision])
versions = subprocess.check_output([str(converter_python), "-m", "pip", "freeze"], text=True)
(run_dir / "converter_versions.txt").write_text(versions, encoding="utf-8")

### 8.4 What should happen when export tools are prepared

On the first run, this stage may take time because it creates an environment and clones/checks out source. On later identical runs, the identity receipts let us reuse the toolchain.

No model quantization happens yet. We are constructing the deterministic machinery that will perform it.

### 8.5 Converting the Merged Checkpoint to GGUF

Hugging Face checkpoints are designed around Python model classes and tensor files. `llama.cpp` uses **GGUF**, a binary format that packages model tensors together with metadata needed by the C/C++ runtime.

The conversion cell calls the pinned `llama.cpp` converter to create an **F16 GGUF** first.

That intermediate is useful for two reasons:

1. format conversion and quantization stay separate operations;
2. if the final Q4 model behaves badly, we can inspect whether the problem appears already in the F16 GGUF.

A receipt binds the conversion to:

- merged-model identity;
- pinned `llama.cpp` revision;
- output hash.

No learning occurs. Tensor values are serialized into the target format at high precision.

### 8.6 Format conversion and quantization are separate sources of deployment risk

Keeping an F16 GGUF intermediate gives us a diagnostic ladder:

$$
\text{HF merged}
\rightarrow
\text{F16 GGUF}
\rightarrow
\text{Q4\_K\_M GGUF}.
$$

If the F16 GGUF produces different tokens from the merged Hugging Face model, we would first inspect conversion, tokenizer metadata or runtime support. If F16 is sound and Q4 changes behavior, quantization becomes the obvious candidate.

Our normal release gate tests the final Q4 artifact because that is what Project 24 actually deploys. The intermediate still gives us a clean place to investigate if a regression appears.

In [ ]:
conversion_dir = run_dir / "conversion"
conversion_dir.mkdir(exist_ok=True)
fp16_gguf = conversion_dir / "qwen-financial-f16.gguf"
fp16_receipt = conversion_dir / "f16_receipt.json"
conversion_identity = {**merge_identity, "llama_revision": llama_revision}
saved = json.loads(fp16_receipt.read_text(encoding="utf-8")) if fp16_receipt.exists() else {}
reuse_fp16 = (fp16_gguf.is_file() and saved.get("identity") == conversion_identity
              and saved.get("sha256") == sha256(fp16_gguf))
if not reuse_fp16:
    partial = fp16_gguf.with_suffix(".gguf.partial")
    run_visible([converter_python, llama_dir / "convert_hf_to_gguf.py", merged_dir,
                 "--outfile", partial, "--outtype", "f16"])
    partial.replace(fp16_gguf)
    save_json(fp16_receipt, {"identity": conversion_identity, "sha256": sha256(fp16_gguf)})
else:
    log("Verified F16 conversion reused")

### 8.7 What the F16 GGUF gives us

After this cell, we should have one large F16 GGUF and its hash/receipt.

It is not yet the small local artifact intended for Project 24. F16 stores roughly two bytes per scalar weight before metadata/architecture details, so a 2B-class model is several gigabytes. The Q4_K_M stage will reduce that sharply.

### 8.8 Building the Exact Local Runtime

We compile two CPU-only `llama.cpp` binaries from the pinned source revision:

- `llama-quantize`;
- `llama-server`.

CUDA is disabled for this build step. That doesn't mean Project 24 can never offload layers to a GPU; it means the export-validation build is reproducible and CPU-capable.

Why compile instead of downloading an arbitrary binary? GGUF support, model architectures, tokenizer behavior and server APIs evolve. A pinned source revision gives the manifest an exact runtime identity.

The build receipt records the source revision and build mode. Later, the quantized artifact is tested against this known runtime rather than against whatever `llama.cpp` happens to be installed globally.

In [ ]:
build_dir = llama_dir / "build-cpu"
quantizer = build_dir / "bin/llama-quantize"
llama_server = build_dir / "bin/llama-server"
build_receipt = tools_dir / "cpu_build.json"
build_identity = {"revision": llama_revision, "cuda": False, "server": validate_gguf}
saved = json.loads(build_receipt.read_text(encoding="utf-8")) if build_receipt.exists() else {}
if saved != build_identity or not quantizer.is_file() or (validate_gguf and not llama_server.is_file()):
    log("Building CPU export tools", workers=min(4, os.cpu_count() or 2))
    run_visible(["cmake", "-S", llama_dir, "-B", build_dir, "-DCMAKE_BUILD_TYPE=Release",
                 "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF"])
    targets = ["llama-quantize", "llama-server"] if validate_gguf else ["llama-quantize"]
    run_visible(["cmake", "--build", build_dir, "--config", "Release", "-j", str(min(4, os.cpu_count() or 2)), "--target", *targets])
    save_json(build_receipt, build_identity)
else:
    log("CPU export tools reused")

### 8.9 What we expect from the build stage

The first run should configure and compile the required binaries. Later runs can reuse them when the receipt matches.

Compilation errors belong to deployment engineering, not model training. Keeping the stage separate makes that boundary visible.

### 8.10 Q4_K_M Quantization

The F16 GGUF is now quantized to **Q4_K_M**.

Quantization replaces high-precision weights with low-bit representations plus per-block scaling information. In a simple symmetric scalar picture,

$$
w \approx s q,
$$

where $q$ is a small integer code and $s$ is a scale chosen for a block of weights.

Real `Q4_K_M` is more sophisticated than one global 4-bit grid. It uses block/group quantization and a mixed strategy for model tensors. We therefore should not describe the file as “every parameter is exactly four independent bits.” The useful economic fact is that it produces a much smaller local model while trying to preserve the behavior of the merged 16-bit checkpoint.

### Quantization is lossy

Let $W$ be a merged weight matrix and $\hat W$ its quantized approximation:

$$
\hat W = W + \varepsilon_Q.
$$

The error $\varepsilon_Q$ changes logits slightly. Most decisions may remain the same, but a token near a decision boundary can change. Structured JSON generation can be sensitive to exactly such local shifts.

That is why we **don't** assume:

$$
\text{adapter passed}
\Rightarrow
\text{quantized model passed}.
$$

It validates the final GGUF separately.

### Training versus deployment one last time

The sequence is:

$$
\text{16-bit Qwen base}
\overset{\text{LoRA SFT}}{\longrightarrow}
\text{base + FP32 adapter}
\overset{\text{merge}}{\longrightarrow}
\text{16-bit merged model}
\overset{\text{GGUF}}{\longrightarrow}
\text{F16 GGUF}
\overset{\text{Q4\_K\_M}}{\longrightarrow}
\text{local quantized model}.
$$

Calling the original optimization “4-bit training” would collapse these distinct operations and describe the method incorrectly.

### 8.11 Quantization error propagates through a deep autoregressive model

A small weight perturbation in one layer creates an activation perturbation. Later layers transform that perturbation, and autoregressive generation can amplify it because a changed token changes the entire future prefix.

If the unquantized next-token logits are $z$ and quantization produces

$$
\hat z=z+\delta z,
$$

the argmax token changes when the perturbation crosses the margin between the best and second-best logits.

Let

$$
m=z_{(1)}-z_{(2)}
$$

be that margin. A token with a large $m$ is robust to small $\delta z$. A low-margin decision is easier to flip.

This is why exact output equality between F16 and Q4 isn't the right deployment requirement. We care whether the quantized model continues to satisfy the analyst contract across representative tasks.

### Why a 1.3 GB deployment file can still represent a 2B model

Low-bit quantization compresses the dominant dense tensors dramatically. The final file also contains metadata and some tensors may use different quantization treatment, so file size doesn't equal

$$
2\text{B}\times 4\text{ bits}
$$

exactly.

The useful engineering interpretation is that Q4_K_M moves a 2B-class analyst into a size range suitable for local laptop inference, while post-quantization tests determine whether the compression is acceptable.

### Quantization doesn't make the model knowledgeable about new data

Quantization is a representation change, not additional training. The Q4 model knows exactly the same *type* of adapted behavior as the merged model, up to approximation error. It doesn't gain newer financial facts during conversion.

Freshness still comes from Project 24's data and retrieval layer.

In [ ]:
export_dir = run_dir / "export"
export_dir.mkdir(exist_ok=True)
gguf_path = export_dir / "qwen-financial-Q4_K_M.gguf"
quantization_receipt = export_dir / "quantization.json"
saved = json.loads(quantization_receipt.read_text(encoding="utf-8")) if quantization_receipt.exists() else {}
reuse_quantization = (gguf_path.is_file() and saved.get("identity") == conversion_identity
                      and saved.get("sha256") == sha256(gguf_path))
if not reuse_quantization:
    partial = gguf_path.with_suffix(".gguf.partial")
    run_visible([quantizer, fp16_gguf, partial, "Q4_K_M"])
    partial.replace(gguf_path)
    save_json(quantization_receipt, {"identity": conversion_identity, "sha256": sha256(gguf_path)})
else:
    log("Verified Q4_K_M quantization reused")
with gguf_path.open("rb") as stream:
    if stream.read(4) != b"GGUF":
        raise ValueError("The output is not a GGUF file.")
model_sha = sha256(gguf_path)
log("Local model exported", path=gguf_path, gib=round(gguf_path.stat().st_size / 2**30, 3))

### 8.12 Quantization integrity checks

The cell verifies the GGUF magic bytes, computes the model hash and records file size.

The hash is especially important because Project 24 later prints the exact deployed model SHA. We can then prove that the model being analyzed is the same binary that passed post-quantization checks.

File size is an engineering metric, not a quality metric. A smaller artifact is useful only if the behavior survives.

### 8.13 Testing the Model We Will Actually Deploy

The post-quantization validation starts a local `llama-server` against the Q4_K_M GGUF. It runs on CPU in this export test (`--ngl 0`) with the 8,192-token training context and Jinja chat templating.

We take one held-out example from each of the five task families. The server receives OpenAI-compatible chat requests with:

- deterministic temperature;
- fixed seed;
- JSON-schema response format;
- thinking disabled.

The resulting text passes through the **same schema, evidence, numerical and repetition checks** used for the adapter.

If an answer fails, one repair attempt is allowed. If any task still fails, export is rejected.

### Why only five GGUF examples after thirty adapter checks?

The two suites answer different questions.

The 30-case adapter suite asks: **did fine-tuning produce a behaviorally acceptable adapter across the held-out task sample?**

The 5-case GGUF suite asks: **did merge, conversion, quantization and local runtime obviously break that behavior across any task family?**

The second test is a deployment regression smoke test rather than a replacement for the larger adapter evaluation.

### Server-side constrained generation

Supplying a JSON schema at generation time narrows the space of legal outputs. In probabilistic terms, unconstrained decoding chooses among the whole vocabulary:

$$
x_{t+1}\in\mathcal V.
$$

A grammar/schema-constrained decoder restricts the allowed set based on the partial JSON state:

$$
x_{t+1}\in\mathcal V_t^{\text{valid}}\subseteq\mathcal V.
$$

The model still determines probabilities inside that legal set. Constraints don't decide which claim is financially sensible. They prevent syntactically illegal structures and invalid enum/reference forms.

This separation recurs in Project 24: generation constraints handle structure; validators handle traceability; financial interpretation still needs evaluation.

### 8.14 Local inference: prompt processing and autoregressive generation have different costs

When `llama-server` receives a request, it first processes the complete prompt. This **prefill** phase can operate on many prompt tokens in parallel.

Generation then produces tokens sequentially:

$$
y_1\rightarrow y_2\rightarrow \cdots\rightarrow y_G.
$$

Token $y_{g+1}$ cannot be finalized until $y_g$ exists, so decode throughput is usually much lower than prompt-token throughput.

Project 24 later measures exactly this difference on the user's local machine. A 6,000-token evidence packet may be processed at hundreds of tokens per second, while new output tokens arrive around single-digit tokens per second.

This is one reason the final analyst keeps responses concise and structured. Retrieval can spend thousands of tokens on evidence, but generating unnecessary prose is expensive on local hardware.

### Context length is a budget shared by prompt and answer

If runtime context is $C$, prompt tokens $P$ and reserved answer tokens $G$ have to satisfy

$$
P+G\le C.
$$

Project 24 later raises the runtime context beyond the 8,192-token training maximum, but still audits token budgets explicitly.

Long context shouldn't be treated as free memory. More prompt text increases prefill time and can dilute relevance. RAG should retrieve *enough* evidence, not everything available.

In [ ]:
import socket

import requests

gguf_rows = [next(row for row in acceptance_rows if row["task"] == task) for task in tasks]
gguf_checks_path = run_dir / "gguf_checks.json"
gguf_identity = {"sha256": model_sha, "llama_revision": llama_revision, "context": training_context,
                 "generation_tokens": generation_tokens, "examples": [row["example_id"] for row in gguf_rows]}
saved = json.loads(gguf_checks_path.read_text(encoding="utf-8")) if gguf_checks_path.exists() else {}
gguf_checks = saved.get("results", []) if saved.get("identity") == gguf_identity else []
done = {item["example_id"] for item in gguf_checks}
if validate_gguf and len(done) < len(gguf_rows):
    with socket.socket() as socket_probe:
        socket_probe.bind(("127.0.0.1", 0))
        port = socket_probe.getsockname()[1]
    endpoint = f"http://127.0.0.1:{port}"
    with (run_dir / "gguf_server.log").open("w", encoding="utf-8") as server_log:
        server = subprocess.Popen([str(llama_server), "-m", str(gguf_path), "-c", str(training_context),
            "-ngl", "0", "--parallel", "1", "--host", "127.0.0.1", "--port", str(port),
            "--jinja", "--reasoning-format", "none"], stdout=server_log, stderr=subprocess.STDOUT)
        try:
            started = time.monotonic()
            while True:
                if server.poll() is not None:
                    raise RuntimeError("GGUF server exited; inspect gguf_server.log.")
                try:
                    ready = requests.get(endpoint + "/health", timeout=2).ok
                except requests.RequestException:
                    ready = False
                if ready:
                    break
                if time.monotonic() - started > 180:
                    raise TimeoutError("GGUF server did not become ready; inspect gguf_server.log.")
                time.sleep(1)
            for row in gguf_rows:
                if row["example_id"] in done:
                    continue
                messages, attempts = row["messages"][:2], []
                for attempt in range(2):
                    log("Checking quantized model on CPU", task=row["task"], attempt=attempt + 1)
                    request = {"messages": messages, "temperature": 0, "seed": seed, "max_tokens": generation_tokens,
                               "chat_template_kwargs": {"enable_thinking": False},
                               "response_format": {"type": "json_schema", "json_schema": {"name": "analysis", "schema": output_schema(row)}}}
                    response = requests.post(endpoint + "/v1/chat/completions", json=request, timeout=600)
                    response.raise_for_status()
                    choice = response.json()["choices"][0]
                    result = decode_result(choice["message"]["content"], row)
                    if choice["finish_reason"] != "stop":
                        result["errors"].append("GGUF generation did not end normally")
                    attempts.append(result)
                    if not result["errors"]:
                        break
                    messages = repair_messages(row, result)
                gguf_checks.append({"example_id": row["example_id"], "task": row["task"], "attempts": attempts, **attempts[-1]})
                save_json(gguf_checks_path, {"identity": gguf_identity, "results": gguf_checks})
        finally:
            server.terminate()
            try:
                server.wait(timeout=20)
            except subprocess.TimeoutExpired:
                server.kill()
                server.wait()
if validate_gguf:
    display(pd.DataFrame(gguf_checks)[["task", "schema", "errors"]])
    if any(item["errors"] for item in gguf_checks):
        raise ValueError("Quantized checks failed. The model and full outputs are saved; inspect gguf_checks.json.")

### 8.15 What the final GGUF checks should show

A successful run should give five passing cases, ideally on the first attempt, one for each task.

If the adapter passed but the GGUF fails consistently, we would investigate:

- quantization sensitivity;
- converter/runtime revision;
- chat-template transfer;
- context handling;
- schema support in the local server.

We would not retrain immediately. The staged pipeline lets us locate the regression first.

The source notebook doesn't contain saved outputs for this stage, so we don't claim a pass count here. Project 24 later reads a published manifest that *does* report the checks for its deployed artifact, and we can analyze those actual results there.

### 8.16 The Export Manifest and Handoff to Project 24

The final cell writes a manifest containing:

- base model and pinned revision;
- adapter hash;
- quantized filename and hash;
- quantization type;
- training context;
- `llama.cpp` revision;
- chat-template hash;
- thinking setting;
- adapter-check status;
- GGUF-check status.

It also copies the output schema and dataset manifest, writes SHA-256 checksums and preserves the chat template.

We can now identify the finished analyst as a chain of immutable inputs:

$$
\text{model identity}
=
H(
\text{base revision},
\text{dataset},
\text{recipe},
\text{adapter},
\text{merge},
\text{converter},
\text{quantized file},
\text{template}
).
$$

That expression is conceptual rather than one literal hash operation, but it captures the engineering goal. “The Qwen model” is too vague for a reproducible financial system. We want to know exactly which weights, data, schema and runtime produced an answer.

The exported artifacts have different roles:

- **adapter:** compact learned task delta;
- **merged weights:** high-precision base plus delta;
- **Q4_K_M GGUF:** local deployment model;
- **training log/state:** optimization record;
- **adapter checks:** behavior gate before merge;
- **GGUF checks:** post-quantization regression gate;
- **manifest/checksums:** provenance.

This is the handoff point to Project 24. Fine-tuning has taught a response discipline. The application still has to obtain the right market, filing, macro and event information; decide what is available as of the requested date; retrieve relevant documents; compress everything into a finite context; call the local model; and reject answers that violate the evidence contract.

#### Reproducibility should include failure records

If a run aborts because of non-finite loss, acceptance failure or GGUF regression, that result is useful too. Keeping the logs and receipts avoids rerunning the same broken recipe without knowing why it failed.

A release pipeline becomes trustworthy when successful artifacts and blocked artifacts are both explainable.

#### The exported model is a release, not simply the last training state

A useful operational mindset is to treat the GGUF like a software release.

Training can produce many checkpoints. Only one adapter passes acceptance, merges successfully, converts successfully, quantizes successfully and passes the final runtime checks. The manifest identifies that released artifact.

If we later retrain with new examples, the new adapter and GGUF should receive new hashes and a new release record. We should not overwrite the old identity and pretend the system was always the newer model.

That lets Project 24 answers remain reproducible even after future model improvements.

### 8.17 What fine-tuning contributes to the final analyst — and what it deliberately leaves outside

After this training pipeline, the Qwen component has learned a stronger **response policy**. It has seen examples of how to:

- separate fact, interpretation and uncertainty;
- cite packet evidence;
- respect a structured answer contract;
- keep factual numbers traceable;
- produce concise financial-analysis fields.

It still doesn't decide which current SEC filing should be read, reconstruct quarterly financial statements, compute yield-curve changes, calculate market risk, identify the latest CPI release, or search a document archive.

Those functions belong to ordinary finance/data code in Project 24.

We can express the deployment answer as

$$
y
=
\operatorname{LLM}_{\theta+\Delta\theta_{\text{LoRA}}}
\left(
q,\,
E_{\text{retrieved}},\,
C_{\text{structured}},\,
I_{\text{system}}
\right),
$$

where $q$ is the question, $E$ is retrieved source evidence, $C$ is calculated context and $I$ contains the analyst rules/schema.

The model is therefore **downstream of the evidence architecture**. A fine-tuned analyst with stale or incorrectly dated evidence can still produce a confidently wrong answer. A strong evidence architecture with an unadapted small model can still produce structurally poor or untraceable text. We built the two layers separately so each can be tested.

Project 24 now picks up from this exact handoff.

In [ ]:
export_manifest = {"base_model": base_model, "base_revision": base_revision, "adapter_sha256": trained["adapter_sha256"],
                   "filename": gguf_path.name, "sha256": model_sha, "quantization": "Q4_K_M",
                   "training_context": training_context, "llama_cpp_revision": llama_revision,
                   "chat_template_sha256": chat_template_hash, "enable_thinking": False,
                   "adapter_checks_passed": adapter_passed, "gguf_generation_checked": validate_gguf,
                   "gguf_checks_passed": all(not item["errors"] for item in gguf_checks) if validate_gguf else None}
save_json(export_dir / "export_manifest.json", export_manifest)
save_json(export_dir / "output_schema.json", Analysis.model_json_schema())
save_json(export_dir / "dataset_manifest.json", manifest)
(export_dir / "SHA256SUMS").write_text(f"{model_sha}  {gguf_path.name}\n", encoding="utf-8")
(export_dir / "chat_template.jinja").write_text(chat_template, encoding="utf-8")
display(pd.DataFrame([{"artifact": label, "path": str(path)} for label, path in
                     [("LoRA adapter", adapter_dir), ("merged weights", merged_dir), ("Q4_K_M model", gguf_path),
                      ("training log", run_dir / "training_log.jsonl"), ("adapter checks", checks_path),
                      ("export manifest", export_dir / "export_manifest.json")]]))
log("Training and local export complete")
print("Keep the adapter, manifests and quantized model. Checkpoints permit training resumption; the cached base and merge avoid repeat downloads and conversion.")

### 8.18 What a complete successful training run leaves behind

When every stage has run, we should be able to answer four audit questions without rerunning the training workflow:

**What was trained?**  
The manifest identifies the exact Qwen base revision and LoRA recipe.

**On what evidence?**  
The frozen dataset manifest and hashes identify the reviewed chronological split.

**Did the adapted behavior pass?**  
Adapter acceptance records answer that for the larger held-out suite, while the human-review display covers financial judgment qualitatively.

**Is the local deployment file the tested file?**  
The GGUF SHA-256 and post-quantization checks connect the binary to the validation run.

That is enough to treat the model as one versioned component inside a larger analyst system rather than as an opaque file copied into a folder.